# Región Oriente y Región Occidente — 2025 vs 2026 y avance del mes en curso

Ejecute todo en orden (*Run All*). El notebook tiene tres partes:

- **A. Preparación**: parámetros, conexión, los dos Parquet (enero 2025 al último mes cerrado y el del mes en curso
  hasta `CORTE_D`, por defecto ayer) y su lectura.
- **B. Análisis**: todas las tablas y gráficas. Secciones 1 a 9 con la numeración del informe general; 10. avance del
  mes en curso; 11. Foco Res; 12. hallazgos.
- **C. Salidas**: al final se generan los PowerPoint (informe general, top de clientes, un cliente, informe de avance y
  Foco Res por canal de venta) y el Excel del panorama.

# A. Preparación
## A.1 Parámetros

> **Para revisar otro mes de clientes cambie `MES_CLIENTES`** (secciones 9 y 12 y el top de clientes).
> **Mes en curso: `CORTE_D = None` toma hasta ayer** (el último día completo); ponga una fecha para otro corte
> (secciones 10 y 11, informe de avance y Foco Res del mes).

In [ ]:
from datetime import date, timedelta
from pathlib import Path

AQUI = Path.cwd()
SALIDA = AQUI                         # carpeta donde se guardan los PowerPoint y el Excel

# Base: Parquet de enero 2025 a septiembre 2026 (se genera en A.3)
FECHA_INICIO = date(2025, 1, 1)
FECHA_FIN = date(2026, 10, 1)         # EXCLUSIVO
PARQUET = AQUI / f"Descuentos_{FECHA_INICIO:%Y%m%d}_{FECHA_FIN:%Y%m%d}.parquet"
GENERAR_PARQUET = False               # True: lo vuelve a consultar aunque ya exista (toma varios minutos)

# >>> MES A REVISAR EN CLIENTES <<<   1 = enero ... 12 = diciembre; None = el ultimo mes con datos
MES_CLIENTES = 9

CLIENTE = "1200110538"                # cliente del informe individual (C.4)
TOP_CLIENTES = 30                     # clientes en los rankings (top de clientes)
TOP_DISTRIBUIDORAS = 30               # distribuidoras en el top del Foco Res
TOP_RANGOS = 8                        # grupos de material y distribuidoras con grafica de rangos
TOP_PPT = 4                           # grupos de material 
FILAS_PPT = 12                        # filas de la tabla de distribuidoras
TOP_MATERIALES = 10                   # materiales en el informe del cliente

# >>> CORTE DEL MES EN CURSO <<<   ultimo dia incluido; None = ayer (p. ej. date(2026, 10, 11) para otro corte)
CORTE_D = None
GENERAR_PARQUET_D = False             # True: vuelve a consultar el Parquet del mes en curso aunque ya exista
FOCO = ["Molida-Hamburguesa Res", "Res Ref y Cong"]   # grupos de material 6 del Foco Res

## A.2 Librerías y conexión

Usa la conexión `DB_*` del `.env` de esta carpeta o, si no hay, el del proyecto de lista de precios.

In [ ]:
import base64
import io
import os
from contextlib import contextmanager

import matplotlib.pyplot as plt
import pandas as pd
import pyarrow as pa
import pyarrow.parquet as pq
from dotenv import load_dotenv
from IPython.display import HTML, Markdown, display
from sqlalchemy import URL, create_engine, text

pd.set_option("display.max_columns", 50)
pd.set_option("display.width", 200)


def engine():
    load_dotenv(AQUI / ".env") or load_dotenv(AQUI.parent / "Lista de precios" / ".env")
    faltan = [v for v in ("DB_HOST", "DB_NAME", "DB_USER", "DB_PASSWORD") if not os.getenv(v)]
    if faltan:
        raise RuntimeError(f"Faltan en el .env: {', '.join(faltan)}. "
                           f"Copie .env.example como .env en {AQUI} y completelo.")
    url = URL.create("postgresql+psycopg2", username=os.environ["DB_USER"], password=os.environ["DB_PASSWORD"],
                     host=os.environ["DB_HOST"], port=int(os.getenv("DB_PORT", "5432")),
                     database=os.environ["DB_NAME"])
    return create_engine(url, pool_pre_ping=True)


ENGINE = engine()
with ENGINE.connect() as c:
    c.execute(text("SELECT 1"))
print(f"Conexion OK: {ENGINE.url.host}/{ENGINE.url.database}")

## A.3 Generación del Parquet

Consulta `commercial_bi` (vista `daily_sales_sector_10_15_20_21`) de `FECHA_INICIO` a `FECHA_FIN` y escribe el
Parquet por partes de `FILAS_POR_PARTE` filas (son ~10 millones). **Solo consulta si el archivo no existe o si
`GENERAR_PARQUET = True`**; si no, usa el que hay.

- `value` es la venta **neta**; bruta = `value + discount_value`. `discount_value` es el ajuste total.
- Fecha corta = `short_dated_discount`; ajuste de precio de mercado = `discount_value − short_dated_discount`.
- Grano: fecha × cliente × grupo de material 6 × distribuidora × canal × sector. El % de lista y el rango (22
  tramos) se calculan sobre ese grano.

In [ ]:
FILAS_POR_PARTE = 500_000

SQL_DESCUENTOS = """
WITH cliente_attr AS (
    -- Un unico registro por cliente (misma regla que lista de precios:
    -- price_group valido y el menor alfabeticamente).
    SELECT DISTINCT ON (cc.client)
        cc.client,
        NULLIF(cc.price_group, '')       AS price_group,
        cc.name                          AS nombre_cliente,
        NULLIF(cc.customer_typology, '') AS tipologia
    FROM public.commercial_clients cc
    ORDER BY cc.client,
             (cc.price_group IS NULL OR cc.price_group = '') ASC,
             cc.price_group ASC
)
SELECT
    v.date                                           AS fecha,
    v.client                                         AS cliente,
    cp.nombre_cliente,
    cp.price_group                                   AS lista_precio,
    m.gr6                                            AS grupo_material_6,
    g6.name                                          AS nombre_gr6,
    dc.distribution_channel_name                     AS canal_distribucion,
    cdd.new_zone_code                                AS distribuidora,
    regexp_replace(cdd.new_zone_name, '^[^-]*-', '') AS nombre_distribuidora,
    cdd.company_code::text                           AS sociedad,
    -- La tipologia de la venta es la del momento; si viene vacia, la del maestro
    LPAD(COALESCE(NULLIF(v.client_typology, ''), cp.tipologia, '0'), 3, '0') AS tipologia,
    t.name                                           AS nombre_tipologia,
    t.sales_channel_name                             AS canal_venta,
    m.sector,
    SUM(v.kilogram)                                  AS venta_kg,
    SUM(v.value)                                     AS venta_neta,
    SUM(v.value + COALESCE(v.discount_value, 0))     AS venta_bruta,
    SUM(COALESCE(v.discount_value, 0))               AS descuento_total,
    SUM(v.discount_market_price_adjustment)          AS descuento_ajuste_mercado,
    SUM(COALESCE(v.short_dated_discount, 0))         AS descuento_fecha_corta,
    SUM(COALESCE(v.focus_discount, 0))               AS descuento_foco
FROM public.daily_sales_sector_10_15_20_21 v
LEFT JOIN public.materials m
       ON v.material::integer = m.code
LEFT JOIN public.material_groups_6 g6
       ON g6.code = m.gr6
LEFT JOIN public.commercial_distributors_details cdd
       ON v.id_commercial_distributors = cdd.id
LEFT JOIN public.distribution_channels dc
       ON v.distribution_channel = dc.distribution_channel
LEFT JOIN cliente_attr cp
       ON v.client = cp.client
LEFT JOIN public.vw_dim_tipologias_cliente t
       ON t.client_typology_id = LPAD(COALESCE(NULLIF(v.client_typology, ''), cp.tipologia, '0'), 3, '0')
WHERE v.date >= :fecha_inicio
  AND v.date <  :fecha_fin                 -- fin EXCLUSIVO
GROUP BY 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14
HAVING COALESCE(SUM(v.value), 0) <> 0
    OR COALESCE(SUM(v.kilogram), 0) <> 0
ORDER BY fecha, cliente, grupo_material_6;
"""

# Rangos: (limite inferior del % de lista, rotulo), de mayor a menor. El numero del rotulo ordena los tramos.
RANGOS = [(100, "100% O MAS"), (97, "Mas de 97 menos de 100%"), (95, "Mas de 95 menos de 97%"),
          (93, "Mas de 93 menos de 95%"), (90, "Mas de 90 menos de 93%"), (87, "Mas de 87 menos de 90%"),
          (85, "Mas de 85 menos de 87%"), (83, "Mas de 83 menos de 85%"), (80, "Mas de 80 menos de 83%"),
          (77, "Mas de 77 menos de 80%"), (75, "Mas de 75 menos de 77%"), (73, "Mas de 73 menos de 75%"),
          (70, "Mas de 70 menos de 73%"), (67, "Mas de 67 menos de 70%"), (65, "Mas de 65 menos de 67%"),
          (63, "Mas de 63 menos de 65%"), (60, "Mas de 60 menos de 63%"), (57, "Mas de 57 menos de 60%"),
          (55, "Mas de 55 menos de 57%"), (53, "Mas de 53 menos de 55%"), (50, "Mas de 50 menos de 53%")]
_ROTULOS = [f"{i:02d}. {r}" for i, (_, r) in enumerate(RANGOS, start=1)] + [f"{len(RANGOS) + 1:02d}. Menos de 50%"]
_ROTULO_SIN_BRUTA = f"{len(RANGOS) + 2:02d}. Sin venta bruta"
_BORDES = [float("-inf")] + [limite for limite, _ in reversed(RANGOS)] + [float("inf")]
MOTIVOS_FILA = {"descuento_fecha_corta": "Fecha corta", "descuento_ajuste_mercado": "Ajuste mercado",
                "descuento_foco": "Foco"}
COLUMNAS = {
    "fecha": "Fecha", "cliente": "Codigo SAP Cliente", "nombre_cliente": "Nombre Cliente",
    "lista_precio": "Lista de Precios", "venta_kg": "Venta Kg", "venta_neta": "Venta Neta",
    "venta_bruta": "Venta Bruta", "descuento_total": "Valor Total Descuento",
    "descuento_ajuste_mercado": "Descuento Ajuste Precio Mercado",
    "descuento_fecha_corta": "Descuento Rotacion / Fecha Corta", "descuento_foco": "Descuento Foco",
    "pct_descuento_total": "% Descuento Total", "pct_descuento_ajuste": "% Descuento Ajuste Mercado",
    "pct_descuento_fecha_corta": "% Descuento Rotacion / Fecha Corta", "pct_descuento_foco": "% Descuento Foco",
    "precio_promedio": "Precio Promedio", "grupo_material_6": "Grupo Material 6", "nombre_gr6": "Nombre GR6",
    "canal_distribucion": "Canal de Distribucion", "distribuidora": "Distribuidora",
    "nombre_distribuidora": "Nombre Distribuidora", "sociedad": "Sociedad", "tipologia": "Tipologia Cliente",
    "nombre_tipologia": "Nombre Tipologia", "canal_venta": "Canal de Venta", "sector": "Sector",
    "pct_lista": "% Lista Total", "pct_lista_rotacion": "% Lista Dcto Rotacion",
    "pct_lista_ajuste": "% Lista Dcto Ajuste", "pct_lista_foco": "% Lista Dcto Foco",
    "rango": "Rango Total", "rango_rotacion": "Rango Dcto Rotacion", "rango_ajuste": "Rango Dcto Ajuste",
    "rango_foco": "Rango Dcto Foco", "motivo": "Motivo Descuento",
}


def rango(pct_: pd.Series) -> pd.Series:
    """% de lista (0 a 100) -> rotulo del tramo. [97, 100) es 'Mas de 97 menos de 100%'."""
    tramos = pd.cut(pct_, bins=_BORDES, labels=list(reversed(_ROTULOS)), right=False)
    return tramos.astype(str).where(pct_.notna(), _ROTULO_SIN_BRUTA)


def calcular(df: pd.DataFrame) -> pd.DataFrame:
    """Medidas, % sobre la venta bruta, % de lista, rangos y motivo de cada fila."""
    medidas = ["venta_kg", "venta_neta", "venta_bruta", "descuento_total",
               "descuento_ajuste_mercado", "descuento_fecha_corta", "descuento_foco"]
    df[medidas] = df[medidas].astype(float)
    df["precio_promedio"] = (df["venta_neta"] / df["venta_kg"].where(df["venta_kg"] != 0)).round(0)
    bruta = df["venta_bruta"].where(df["venta_bruta"] > 0)   # devoluciones (bruta <= 0): sin %

    def sobre_bruta(serie: pd.Series) -> pd.Series:
        return (serie / bruta * 100).round(2)

    df["pct_descuento_total"] = sobre_bruta(df["descuento_total"])
    df["pct_descuento_ajuste"] = sobre_bruta(df["descuento_ajuste_mercado"])
    df["pct_descuento_fecha_corta"] = sobre_bruta(df["descuento_fecha_corta"])
    df["pct_descuento_foco"] = sobre_bruta(df["descuento_foco"])
    # % de lista: el total usa la neta; los otros aislan un solo descuento, como si fuera el unico
    df["pct_lista"] = sobre_bruta(df["venta_neta"])
    df["pct_lista_rotacion"] = sobre_bruta(df["venta_bruta"] - df["descuento_fecha_corta"])
    df["pct_lista_ajuste"] = sobre_bruta(df["venta_bruta"] - df["descuento_ajuste_mercado"])
    df["pct_lista_foco"] = sobre_bruta(df["venta_bruta"] - df["descuento_foco"])
    for sufijo in ("", "_rotacion", "_ajuste", "_foco"):
        df["rango" + sufijo] = rango(df["pct_lista" + sufijo])
    partes = [df[col].gt(0).map({True: nombre + " + ", False: ""}) for col, nombre in MOTIVOS_FILA.items()]
    texto_ = pd.concat(partes, axis=1).sum(axis=1).str.removesuffix(" + ")
    df["motivo"] = texto_.where(texto_ != "", "Sin descuento")
    df["fecha"] = pd.to_datetime(df["fecha"])
    return df[list(COLUMNAS)].rename(columns=COLUMNAS)


def tipo_parquet(serie: pd.Series) -> pa.DataType:
    if pd.api.types.is_datetime64_any_dtype(serie):
        return pa.timestamp("ms")
    if pd.api.types.is_numeric_dtype(serie):
        return pa.float64()
    return pa.string()


def escribir_parquet(partes, salida: Path) -> int:
    """Escribe parte por parte. El esquema se fija con la primera parte (texto en lo que no es numero ni fecha).
    Si algo falla a mitad, borra el archivo: uno incompleto es peor que ninguno."""
    escritor, filas = None, 0
    try:
        for df in partes:
            if escritor is None:
                esquema = pa.schema([(c, tipo_parquet(df[c])) for c in df.columns])
                escritor = pq.ParquetWriter(salida, esquema)
            escritor.write_table(pa.Table.from_pandas(df, schema=esquema, preserve_index=False))
            filas += len(df)
            print(f"  {filas:,} filas")
    except BaseException:
        if escritor is not None:
            escritor.close()
            salida.unlink(missing_ok=True)
        raise
    if escritor is not None:
        escritor.close()
    return filas


if GENERAR_PARQUET or not PARQUET.exists():
    print(f"Consultando {FECHA_INICIO} -> {FECHA_FIN} (exclusivo)...")
    with ENGINE.connect() as c:
        c = c.execution_options(stream_results=True)   # el servidor entrega las filas por partes
        partes = pd.read_sql(text(SQL_DESCUENTOS), c, chunksize=FILAS_POR_PARTE,
                             params={"fecha_inicio": FECHA_INICIO, "fecha_fin": FECHA_FIN})
        escribir_parquet((calcular(p) for p in partes), PARQUET)
print(f"Base: {PARQUET.name}")

## A.4 Parquet del mes en curso

Mismo SQL de A.3 desde el 1 del mes anterior hasta `CORTE_D` (incluido), para el avance (mes anterior cerrado vs mes
en curso). Si `CORTE_D` es `None`, el corte es ayer. El archivo lleva las fechas en el nombre: un corte nuevo genera uno nuevo. **Si vuelve a correr el mismo
día con el mismo corte, ponga `GENERAR_PARQUET_D = True`** para traer lo que se haya cargado después.

In [ ]:
CORTE_D = CORTE_D or date.today() - timedelta(days=1)          # por defecto, ayer
INICIO_ACT = CORTE_D.replace(day=1)                              # 1 del mes en curso
INICIO_ANT = (INICIO_ACT - timedelta(days=1)).replace(day=1)     # 1 del mes anterior
FIN_D = CORTE_D + timedelta(days=1)                              # EXCLUSIVO
PARQUET_D = AQUI / f"Descuentos_{INICIO_ANT:%Y%m%d}_{FIN_D:%Y%m%d}.parquet"

if GENERAR_PARQUET_D or not PARQUET_D.exists():
    print(f"Consultando {INICIO_ANT} -> {FIN_D} (exclusivo)...")
    with ENGINE.connect() as c:
        c = c.execution_options(stream_results=True)
        partes = pd.read_sql(text(SQL_DESCUENTOS), c, chunksize=FILAS_POR_PARTE,
                             params={"fecha_inicio": INICIO_ANT, "fecha_fin": FIN_D})
        escribir_parquet((calcular(p) for p in partes), PARQUET_D)
print(f"Base del avance: {PARQUET_D.name}")

## A.5 Nombres, formato y funciones comunes

In [ ]:
SOCIEDADES = {"1000": "Región Oriente", "2000": "Región Occidente"}   # company_code: 1000 Región Oriente, 2000 Región Occidente
VISTAS = {"Región Oriente": "Región Oriente", "Región Occidente": "Región Occidente"}
ARCHIVO = {"Región Oriente": "Region_Oriente", "Región Occidente": "Region_Occidente"}   # nombre en los archivos
CORTO = {soc: vista for vista, soc in VISTAS.items()}
NOMBRE_CANAL_VENTA = {"Directo": "Amas de casa"}
CANALES_VENTA_TOP = ["Tradicional", "Institucional", "Modernos", "Amas de casa"]   # orden en tablas y top
ROTULO_CANAL = {"Mayoreo": "Mayoreo", "Puntos Venta": "Puntos de Venta"}   # canal de distribucion
# Canales que siempre facturan al 100% de la lista (se senalan en los hallazgos)
ESPECIALES = ("SUMINISTRO", "GERENCIA", "POLLO CANAL")
# Dimensiones de los rangos (secciones 3 a 6 y 8): rotulo -> columna del Parquet (None = el total)
DIMENSIONES = {"Total": None,
               "Canal de distribución": "Canal de Distribucion",
               "Canal de venta": "Canal de Venta",
               "Grupo de material 6": "Nombre GR6",
               "Distribuidora": "Nombre Distribuidora"}
# Cruce con el canal de distribucion (seccion 7): rotulo -> columna del Parquet
CRUCES = {"Canal de venta": "Canal de Venta"}
TOPE = {"Grupo de material 6": TOP_RANGOS, "Distribuidora": TOP_RANGOS}
CON_CALOR = {"Grupo de material 6"}   # tablas resumen con mapa de calor en los % de ajuste
MEDIDAS = ["Venta Kg", "Venta Bruta", "Venta Neta",
           "Descuento Ajuste Precio Mercado", "Descuento Rotacion / Fecha Corta"]

# --- nombres normalizados
VENTA_BRUTA, VENTA_NETA, VENTA_KG = "Venta bruta ($)", "Venta neta ($)", "Venta kg"
PRECIO_LISTA, PRECIO_NETO = "Precio lista por kg", "Precio neto por kg"
L_GEN, D_GEN = "% aplicado al precio de lista general", "% Ajuste general"
L_AJ, D_AJ = "% aplicado al precio de lista ajuste de precio de mercado", "% Ajuste de precio de mercado"
L_FC, D_FC = "% aplicado al precio de lista fecha corta", "% Ajuste de fecha corta"
P_GEN, P_AJ, P_FC = "Ajuste general ($)", "Ajuste de precio de mercado ($)", "Ajuste de fecha corta ($)"
MILLON = 1     # divisor de los montos: 1 = pesos completos (1e6 = millones)
PILAR = [L_GEN, D_GEN, L_AJ, D_AJ, L_FC, D_FC]
BLOQUE = [VENTA_BRUTA, VENTA_NETA, VENTA_KG, PRECIO_LISTA, PRECIO_NETO,
          L_GEN, D_GEN, P_GEN, L_AJ, D_AJ, P_AJ, L_FC, D_FC, P_FC]
COMPACTO = [VENTA_KG, VENTA_BRUTA, P_GEN, VENTA_NETA, PRECIO_NETO, *PILAR]

# --- colores: motivo y tipo de ajuste comparten color (ajuste = verde, fecha corta = amarillo)
MOTIVOS = {"Con fecha corta": "#eda100", "Solo ajuste de precio de mercado": "#1baf7a", "Sin ajuste": "#b5b4af"}
COLOR_LISTA = {L_GEN: "#52514e", L_AJ: "#1baf7a", L_FC: "#eda100"}
CORTO_LISTA = {L_GEN: "General", L_AJ: "Ajuste de precio de mercado", L_FC: "Fecha corta"}
COLOR_SOC = {"Región Oriente": "#2a78d6", "Región Occidente": "#eb6834"}
TINTE_SOC = {"Región Oriente": "#a9c9f0", "Región Occidente": "#f5b49a"}

TRAMOS = [r[4:] for r in _ROTULOS]  # "100% O MAS", ..., "Menos de 50%"
LIMITE = dict(zip(_ROTULOS, [lim for lim, _ in RANGOS] + [float("-inf")]))
MESES_ES = ["enero", "febrero", "marzo", "abril", "mayo", "junio", "julio", "agosto",
            "septiembre", "octubre", "noviembre", "diciembre"]


# --- formato colombiano
def num(v, dec: int = 1) -> str:
    """Miles con punto, decimales con coma (3.831,3)."""
    if pd.isna(v):
        return "-"
    return f"{v:,.{dec}f}".replace(",", "_").replace(".", ",").replace("_", ".")


def pct(v) -> str:
    return "-" if pd.isna(v) else num(v) + "%"


def pesos(v) -> str:
    return "-" if pd.isna(v) else "$" + num(v, 0)


def variacion(v, en_puntos: bool) -> str:
    """+5,9% (variacion) o -6,4 pts (diferencia entre porcentajes)."""
    if pd.isna(v):
        return "-"
    return ("+" if v >= 0 else "-") + num(abs(v)) + (" pts" if en_puntos else "%")


def formato_de(indicador: str):
    if indicador.startswith("%"):
        return pct
    if indicador.startswith("Venta kg"):   # kilos completos, sin decimales
        return lambda v: num(v, 0)
    if indicador.startswith("Precio") or indicador.endswith("($)"):
        return pesos
    return num


# --- indicadores (todo desde sumas, nunca promediando % de filas)
def indicadores(s: pd.DataFrame) -> pd.DataFrame:
    """s: sumas de MEDIDAS (cualquier indice) -> BLOQUE de indicadores."""
    bruta = s["Venta Bruta"]
    b = bruta.where(bruta != 0)
    kg = s["Venta Kg"].where(s["Venta Kg"] != 0)
    aj, fc = s["Descuento Ajuste Precio Mercado"], s["Descuento Rotacion / Fecha Corta"]
    gen = aj + fc
    return pd.DataFrame({
        VENTA_BRUTA: bruta / MILLON,
        VENTA_NETA: s["Venta Neta"] / MILLON,
        VENTA_KG: s["Venta Kg"],
        PRECIO_LISTA: bruta / kg,
        PRECIO_NETO: s["Venta Neta"] / kg,
        L_GEN: (bruta - gen) / b * 100,
        D_GEN: gen / b * 100,
        P_GEN: gen / MILLON,
        L_AJ: (bruta - aj) / b * 100,
        D_AJ: aj / b * 100,
        P_AJ: aj / MILLON,
        L_FC: (bruta - fc) / b * 100,
        D_FC: fc / b * 100,
        P_FC: fc / MILLON,
    }, index=s.index)


def comparar(t: pd.DataFrame) -> pd.DataFrame:
    """t: filas = años -> filas = indicadores; columnas 2025, 2026 y Variación (% o pts)."""
    t = t.T
    out = t[[A1, A2]].copy()
    out.columns = [str(A1), str(A2)]
    es_pct = out.index.str.startswith("%")
    out["Variación"] = ((out[str(A2)] / out[str(A1)] - 1) * 100).where(~es_pct, out[str(A2)] - out[str(A1)])
    return out


def es_variacion(col) -> bool:
    """Columna de variación entre años: 'Variación' en cualquier nivel del encabezado."""
    return any(str(x).startswith("Variación") for x in (col if isinstance(col, tuple) else (col,)))


def estilo_filas(t: pd.DataFrame, titulo: str):
    """Tabla con indicadores en las filas: formato por fila; las columnas 'Variación' en % o pts."""
    variaciones = [c for c in t.columns if es_variacion(c)]
    valores = [c for c in t.columns if c not in variaciones]
    est = t.style
    for fila in t.index:
        est = est.format(formato_de(fila), subset=pd.IndexSlice[[fila], valores])
        if variaciones:
            est = est.format(lambda v, p=fila.startswith("%"): variacion(v, p), subset=pd.IndexSlice[[fila], variaciones])
    return est.set_caption(titulo)


def estilo_columnas(t: pd.DataFrame, titulo: str):
    """Tabla con indicadores en las columnas (nivel 0 si son varias): formato por columna."""
    est = t.style
    for c in t.columns:
        if pd.api.types.is_numeric_dtype(t[c]):   # las columnas de texto (p. ej. canal de venta) van tal cual
            est = est.format(formato_de(c[0] if isinstance(c, tuple) else c), subset=[c])
    return est.set_caption(titulo)


def lado_a_lado(fig, estilo):
    """Grafica y tabla una al lado de la otra (se apilan si no caben)."""
    buf = io.BytesIO()
    fig.savefig(buf, format="png", dpi=100, bbox_inches="tight", facecolor="white")
    plt.close(fig)
    img = base64.b64encode(buf.getvalue()).decode()
    display(HTML('<div style="display:flex;gap:18px;align-items:flex-start;flex-wrap:wrap">'
                 f'<img style="max-width:62%;min-width:520px" src="data:image/png;base64,{img}"/>'
                 f'<div style="font-size:11px">{estilo.to_html()}</div></div>'))


def ejes_limpios(ax, formato_y=None, formato_x=None):
    ax.grid(axis="y" if formato_y else "x", color="#e6e5e1", linewidth=0.8)
    ax.set_axisbelow(True)
    for lado in ("top", "right"):
        ax.spines[lado].set_visible(False)
    ax.tick_params(colors="#52514e", labelsize=8)
    if formato_y:
        ax.yaxis.set_major_formatter(formato_y)
    if formato_x:
        ax.xaxis.set_major_formatter(formato_x)

## A.6 Lectura del Parquet

Una sola pasada por lotes (~10 millones de filas) que deja tablas pequeñas:

- `mensual_soc`: sociedad × mes, **todos los meses** (panorama y evolución).
- `cubo`: dimensión × miembro × sociedad × año × rango × motivo, **enero al último mes con datos** (rangos).
- `cruce`: canal de venta por canal de distribución (sección 7).
- `clientes` y `clientes_det`: **solo el mes de `MES_CLIENTES`** y el mismo mes del año anterior (sección 9 y top
  de clientes).

La sociedad sale de `company_code`: si el Parquet no trae la columna `Sociedad`, se asigna por distribuidora.

In [ ]:
archivo = pq.ParquetFile(PARQUET)
# Ultimo dia con datos (de las estadisticas del archivo, sin leerlo)
i_fecha = archivo.schema_arrow.get_field_index("Fecha")
FECHA_MAX = pd.Timestamp(max(archivo.metadata.row_group(i).column(i_fecha).statistics.max
                             for i in range(archivo.metadata.num_row_groups)))
A2 = FECHA_MAX.year
A1 = A2 - 1
ANIOS = [A1, A2]
MES_ULTIMO = FECHA_MAX.month                         # cierre del acumulado
MESES_ACUM = list(range(1, MES_ULTIMO + 1))
PERIODO = f"ene–{MESES_ES[MES_ULTIMO - 1][:3]}"      # "ene–sep"
MES_FIN_TXT = MESES_ES[MES_ULTIMO - 1]               # "septiembre"
MES_CLI = MES_CLIENTES or MES_ULTIMO                 # mes de los clientes (A.1)
MES_TXT = MESES_ES[MES_CLI - 1]

columnas = ["Fecha", "Distribuidora", *MEDIDAS, "Rango Total", "Codigo SAP Cliente", "Nombre Cliente",
            *[c for c in DIMENSIONES.values() if c]]
if "Sociedad" in archivo.schema_arrow.names:
    columnas.append("Sociedad")
    sociedad_de = None
else:
    with ENGINE.connect() as c:
        sociedad_de = pd.read_sql(text(
            "SELECT DISTINCT new_zone_code, company_code::text AS sociedad "
            "FROM public.commercial_distributors_details"), c).set_index("new_zone_code")["sociedad"]
    assert sociedad_de.index.is_unique, "Hay distribuidoras en mas de una sociedad"

partes_mes, partes_cubo, partes_cli, nombres, partes_cv, partes_cruce, partes_det = [], [], [], [], [], [], []
for lote in archivo.iter_batches(batch_size=FILAS_POR_PARTE, columns=columnas):
    d = lote.to_pandas()
    if sociedad_de is not None:
        d["Sociedad"] = d["Distribuidora"].map(sociedad_de)
    d["Sociedad"] = d["Sociedad"].map(SOCIEDADES).fillna(d["Sociedad"]).fillna("Sin sociedad")
    d = d[d["Sociedad"].isin(VISTAS.values())]
    d["Canal de Venta"] = d["Canal de Venta"].replace(NOMBRE_CANAL_VENTA)
    d["Mes"] = d["Fecha"].dt.to_period("M")
    partes_mes.append(d.groupby(["Sociedad", "Mes"])[MEDIDAS].sum())

    d["Año"] = d["Fecha"].dt.year
    d["Motivo"] = (pd.Series("Sin ajuste", index=d.index)
                   .mask(d["Descuento Ajuste Precio Mercado"] > 0, "Solo ajuste de precio de mercado")
                   .mask(d["Descuento Rotacion / Fecha Corta"] > 0, "Con fecha corta"))

    acum = d[d["Fecha"].dt.month.isin(MESES_ACUM) & d["Año"].isin(ANIOS)]
    for dim, col in DIMENSIONES.items():
        miembro = (acum[col].fillna("(vacío)") if col else pd.Series("Total", index=acum.index)).rename("Miembro")
        partes_cubo.append(pd.concat(
            {dim: acum.groupby([miembro, "Sociedad", "Año", "Rango Total", "Motivo"])[MEDIDAS].sum()},
            names=["Dimension"]))

    for dim, col in CRUCES.items():
        partes_cruce.append(pd.concat(
            {dim: acum.groupby([acum[col].fillna("(vacío)").rename("Miembro"), "Canal de Distribucion",
                                "Sociedad", "Año"])[MEDIDAS].sum()}, names=["Dimension"]))

    mes_cli = d[(d["Fecha"].dt.month == MES_CLI) & d["Año"].isin(ANIOS)]
    partes_cli.append(mes_cli.groupby(["Codigo SAP Cliente", "Sociedad", "Canal de Distribucion", "Año"])[MEDIDAS].sum())
    nombres.append(mes_cli.groupby("Codigo SAP Cliente")["Nombre Cliente"].first())
    partes_cv.append(mes_cli[mes_cli["Año"] == A2].groupby(["Codigo SAP Cliente", "Canal de Venta"])["Venta Bruta"].sum())
    partes_det.append(mes_cli[mes_cli["Año"] == A2].fillna({"Canal de Venta": "(sin canal)", "Nombre Distribuidora": "(vacío)"})
                      .groupby(["Codigo SAP Cliente", "Sociedad", "Nombre Distribuidora", "Canal de Distribucion",
                                "Canal de Venta"])[MEDIDAS].sum())


def juntar(partes: list) -> pd.DataFrame:
    t = pd.concat(partes)
    return t.groupby(level=list(range(t.index.nlevels))).sum()


mensual_soc = juntar(partes_mes)          # sociedad x mes
cubo = juntar(partes_cubo)
cruce = juntar(partes_cruce)              # canal de venta x canal de distribucion x sociedad x año
clientes = juntar(partes_cli)             # cliente x sociedad x canal de distribucion x año (mes de clientes)
clientes_det = juntar(partes_det)         # cliente x distribuidora x canales (mes de clientes, ultimo año)
nombre_cliente = pd.concat(nombres)
nombre_cliente = nombre_cliente[~nombre_cliente.index.duplicated()]
# canal de venta de cada cliente en el mes (si tuvo varios, el de mas venta bruta)
canal_venta_cliente = (pd.concat(partes_cv).groupby(level=[0, 1]).sum().reset_index()
                       .sort_values("Venta Bruta").drop_duplicates("Codigo SAP Cliente", keep="last")
                       .set_index("Codigo SAP Cliente")["Canal de Venta"])
del partes_mes, partes_cubo, partes_cli, nombres, partes_cv, partes_cruce, partes_det

print(f"Datos hasta {FECHA_MAX:%d/%m/%Y} · acumulado {PERIODO} {A1} vs {A2} · clientes de {MES_TXT} | "
      f"cubo {len(cubo):,} filas · clientes {len(clientes):,} filas")

## A.7 Lectura del Parquet del mes en curso

Las mismas tablas de A.6 con el periodo (mes anterior / mes en curso) en el nivel `Año`, para que las funciones de B y
C sirvan sin cambios: `diario_soc` (sociedad × día), `cubo_d`, `cruce_d`, `clientes_d` y `foco_d` (los materiales de
`FOCO` por canal de venta, canal de distribución, distribuidora, cliente, rango y motivo).

In [ ]:
archivo_d = pq.ParquetFile(PARQUET_D)
i_fecha = archivo_d.schema_arrow.get_field_index("Fecha")
FECHA_MAX_D = pd.Timestamp(max(archivo_d.metadata.row_group(i).column(i_fecha).statistics.max
                               for i in range(archivo_d.metadata.num_row_groups)))
MES_ANT_TXT, MES_ACT_TXT = MESES_ES[INICIO_ANT.month - 1], MESES_ES[INICIO_ACT.month - 1]
ETQ_ANT = f"{MES_ANT_TXT[:3].capitalize()} {INICIO_ANT.year}"          # "Sep 2026"
ETQ_ACT = f"{MES_ACT_TXT[:3].capitalize()} al {FECHA_MAX_D.day}"       # "Oct al 13"
PERIODOS_D = [ETQ_ANT, ETQ_ACT]

con_sociedad = "Sociedad" in archivo_d.schema_arrow.names
columnas_d = [c for c in columnas if c != "Sociedad"] + (["Sociedad"] if con_sociedad else [])
if not con_sociedad and sociedad_de is None:
    with ENGINE.connect() as c:
        sociedad_de = pd.read_sql(text(
            "SELECT DISTINCT new_zone_code, company_code::text AS sociedad "
            "FROM public.commercial_distributors_details"), c).set_index("new_zone_code")["sociedad"]

partes_dia, partes_cubo, partes_cruce, partes_cli, nombres, partes_cv, partes_foco = [], [], [], [], [], [], []
for lote in archivo_d.iter_batches(batch_size=FILAS_POR_PARTE, columns=columnas_d):
    d = lote.to_pandas()
    if not con_sociedad:
        d["Sociedad"] = d["Distribuidora"].map(sociedad_de)
    d["Sociedad"] = d["Sociedad"].map(SOCIEDADES).fillna(d["Sociedad"]).fillna("Sin sociedad")
    d = d[d["Sociedad"].isin(VISTAS.values())]
    d["Canal de Venta"] = d["Canal de Venta"].replace(NOMBRE_CANAL_VENTA)
    d["Dia"] = d["Fecha"].dt.to_period("D")
    partes_dia.append(d.groupby(["Sociedad", "Dia"])[MEDIDAS].sum())

    # el nivel "Año" lleva el periodo: mes anterior cerrado o mes en curso
    d["Año"] = pd.Series(ETQ_ANT, index=d.index).mask(d["Fecha"] >= pd.Timestamp(INICIO_ACT), ETQ_ACT)
    d["Motivo"] = (pd.Series("Sin ajuste", index=d.index)
                   .mask(d["Descuento Ajuste Precio Mercado"] > 0, "Solo ajuste de precio de mercado")
                   .mask(d["Descuento Rotacion / Fecha Corta"] > 0, "Con fecha corta"))
    for dim, col in DIMENSIONES.items():
        miembro = (d[col].fillna("(vacío)") if col else pd.Series("Total", index=d.index)).rename("Miembro")
        partes_cubo.append(pd.concat(
            {dim: d.groupby([miembro, "Sociedad", "Año", "Rango Total", "Motivo"])[MEDIDAS].sum()},
            names=["Dimension"]))
    for dim, col in CRUCES.items():
        partes_cruce.append(pd.concat(
            {dim: d.groupby([d[col].fillna("(vacío)").rename("Miembro"), "Canal de Distribucion",
                             "Sociedad", "Año"])[MEDIDAS].sum()}, names=["Dimension"]))
    partes_cli.append(d.groupby(["Codigo SAP Cliente", "Sociedad", "Canal de Distribucion", "Año"])[MEDIDAS].sum())
    nombres.append(d.groupby("Codigo SAP Cliente")["Nombre Cliente"].first())
    partes_cv.append(d[d["Año"] == ETQ_ACT].groupby(["Codigo SAP Cliente", "Canal de Venta"])["Venta Bruta"].sum())
    f = d[d["Nombre GR6"].isin(FOCO)].fillna({"Canal de Venta": "(sin canal)", "Nombre Distribuidora": "(vacío)",
                                               "Canal de Distribucion": "(vacío)"})
    partes_foco.append(f.groupby(["Nombre GR6", "Sociedad", "Año", "Canal de Venta", "Nombre Distribuidora",
                                  "Canal de Distribucion", "Codigo SAP Cliente", "Rango Total", "Motivo"])[MEDIDAS].sum())

diario_soc = juntar(partes_dia)           # sociedad x dia
cubo_d = juntar(partes_cubo)
cruce_d = juntar(partes_cruce)
clientes_d = juntar(partes_cli)           # cliente x sociedad x canal de distribucion x periodo
foco_d = juntar(partes_foco)              # material x sociedad x periodo x canales x distribuidora x cliente x rango
                                          # x motivo
nombre_cliente_d = pd.concat(nombres)
nombre_cliente_d = nombre_cliente_d[~nombre_cliente_d.index.duplicated()]
canal_venta_cliente_d = (pd.concat(partes_cv).groupby(level=[0, 1]).sum().reset_index()
                         .sort_values("Venta Bruta").drop_duplicates("Codigo SAP Cliente", keep="last")
                         .set_index("Codigo SAP Cliente")["Canal de Venta"])
del partes_dia, partes_cubo, partes_cruce, partes_cli, nombres, partes_cv, partes_foco

faltan = [m for m in FOCO if m not in foco_d.index.unique("Nombre GR6")]
assert not faltan, f"Sin venta en el periodo para: {faltan} (revise el nombre en FOCO)"
print(f"Datos hasta {FECHA_MAX_D:%d/%m/%Y} · {ETQ_ANT} vs {ETQ_ACT} | cubo {len(cubo_d):,} filas · "
      f"clientes {len(clientes_d):,} filas · foco {len(foco_d):,} filas")

# B. Análisis

Secciones 1 a 9 con la numeración del informe general (C.2); 10 y 11, el avance del mes en curso y el Foco Res (C.6 y C.7); 12, los hallazgos. Los % se calculan siempre desde las sumas (nunca promediando %
de filas); % con un decimal, precios, montos en pesos y kg completos, sin decimales.

## 1. Contexto del alcance

*Inicialmente el foco fue el ajuste de fecha corta, pero fruto del primer modelo análisis de datos, se incorporó
al alcance el porcentaje de cumplimiento de listas, con el ajuste general: en torno al ajuste de precio de mercado y
fecha corta.*

- **Venta bruta**: lo que se habría cobrado a precio de lista (venta neta + ajustes). Es la base de todas las
  participaciones.
- **Venta neta**: lo facturado después de ajustes; se muestra para validar.
- **Venta kg**: kilos vendidos.
- **Rangos**: 22 tramos del % aplicado al precio de lista general, de “100% o más” a “menos de 50%”. Cada venta
  (día × cliente × grupo de material × distribuidora) cae en uno; las devoluciones (bruta ≤ 0) no tienen rango.
- **Tipos de ajuste**: general = ajuste de precio de mercado + fecha corta; % aplicado al precio de lista + % de
  ajuste = 100%.

  | % aplicado al precio de lista | % de ajuste | Cálculo |
  |---|---|---|
  | general | % Ajuste general | neta / bruta |
  | ajuste de precio de mercado | % Ajuste de precio de mercado | (bruta − ajuste) / bruta: solo ese ajuste |
  | fecha corta | % Ajuste de fecha corta | (bruta − fecha corta) / bruta: solo ese ajuste |

- **Periodos**: acumulado enero a septiembre, 2025 vs 2026; la evolución mes a mes incluye octubre a diciembre
  de 2025; los clientes, solo el mes de `MES_CLIENTES` contra el mismo mes de 2025.

El canal de venta *Directo* se muestra como *Amas de casa*.

## 2. Panorama 2025–2026

Acumulado enero a septiembre por sociedad y, debajo, la evolución mes a mes (todos los meses, incluido
octubre–diciembre de 2025): venta kg, venta bruta y neta (la franja es el ajuste), precio de lista y neto por kg
y los tres % aplicados al precio de lista. La línea punteada separa los años.

In [ ]:
def acumulado(soc) -> pd.DataFrame:
    """Sumas por año del acumulado enero..ultimo mes (filas: año)."""
    m = mensual_soc[mensual_soc.index.get_level_values("Sociedad") == soc]
    meses_ = m.index.get_level_values("Mes")
    m = m[meses_.month.isin(MESES_ACUM) & meses_.year.isin(ANIOS)]
    return m.groupby(m.index.get_level_values("Mes").year).sum()


PANORAMA = {vista: comparar(indicadores(acumulado(soc))) for vista, soc in VISTAS.items()}
panorama = pd.concat(PANORAMA, axis=1)[pd.MultiIndex.from_product([VISTAS, [str(A1), str(A2), "Variación"]])]
estilo_filas(panorama, f"Panorama {PERIODO} {A1} vs {A2}: venta bruta, kg y % aplicado al precio de lista")

In [ ]:
def mensual_de(soc) -> pd.DataFrame:
    m = mensual_soc[mensual_soc.index.get_level_values("Sociedad") == soc].groupby(level="Mes").sum()
    return m[m.index.year.isin(ANIOS)]


MESES_EJE = mensual_soc.index.get_level_values("Mes").unique().sort_values()
MESES_EJE = MESES_EJE[MESES_EJE.year.isin(ANIOS)]
X_MES = list(range(len(MESES_EJE)))
ETIQ_MES = MESES_EJE.strftime("%Y-%m")
CORTE_ANIO = sum(MESES_EJE.year == A1) - 0.5
PANELES = {"kg": "Venta kg por mes",
           "pesos": "Venta bruta y venta neta por mes ($) — la franja es el ajuste",
           "precio": "Precio de lista y precio neto por kg",
           "aplicado": "% aplicado al precio de lista (100% − % de ajuste)"}


def _rotular(ax, serie, formato, arriba=True):
    """Valor del primer y del ultimo mes junto al punto."""
    for i in (0, len(X_MES) - 1):
        v = serie.iloc[i]
        if pd.notna(v):
            ax.annotate(formato(v), (X_MES[i], v), xytext=(0, 6 if arriba else -10), textcoords="offset points",
                        ha="center", va="bottom" if arriba else "top", fontsize=7, color="#0b0b0b")


def figura_evolucion(soc, paneles=tuple(PANELES), figsize=None):
    """Paneles alineados por mes para una sociedad."""
    t = indicadores(mensual_de(soc)).reindex(MESES_EJE)
    color, tinte = COLOR_SOC[soc], TINTE_SOC[soc]
    fig, ejes = plt.subplots(len(paneles), 1, figsize=figsize or (14, 3.2 * len(paneles)), sharex=True, squeeze=False)
    ejes = ejes[:, 0]
    for ax, p in zip(ejes, paneles):
        if p == "kg":
            ax.bar(X_MES, t[VENTA_KG], color=color, width=0.75, edgecolor="white", linewidth=1)
            for i, v in enumerate(t[VENTA_KG]):
                ax.annotate(num(v, 0), (i, v), xytext=(0, 2), textcoords="offset points", ha="center",
                            va="bottom", fontsize=6.5, color="#52514e",
                            rotation=90 if len(X_MES) > 25 else 0)   # muchas barras: rotulo vertical
            ejes_limpios(ax, formato_y=lambda v, _: num(v, 0))
        elif p == "pesos":
            ax.fill_between(X_MES, t[VENTA_NETA], t[VENTA_BRUTA], color=tinte, alpha=0.6, linewidth=0, label="Ajuste")
            ax.plot(X_MES, t[VENTA_BRUTA], color=color, linewidth=2, marker="o", markersize=4, label="Venta bruta")
            ax.plot(X_MES, t[VENTA_NETA], color=color, linewidth=2, linestyle="--", marker="o", markersize=4,
                    label="Venta neta")
            _rotular(ax, t[VENTA_BRUTA], pesos)
            _rotular(ax, t[VENTA_NETA], pesos, arriba=False)
            ejes_limpios(ax, formato_y=lambda v, _: "$" + num(v, 0))
        elif p == "precio":
            ax.plot(X_MES, t[PRECIO_LISTA], color=color, linewidth=2, marker="o", markersize=4, label="Precio lista por kg")
            ax.plot(X_MES, t[PRECIO_NETO], color=color, linewidth=2, linestyle="--", marker="o", markersize=4,
                    label="Precio neto por kg")
            _rotular(ax, t[PRECIO_LISTA], pesos)
            _rotular(ax, t[PRECIO_NETO], pesos, arriba=False)
            ejes_limpios(ax, formato_y=lambda v, _: pesos(v))
        elif p == "aplicado":
            for col, c_ in COLOR_LISTA.items():
                ax.plot(X_MES, t[col], color=c_, linewidth=2, marker="o", markersize=4, label=CORTO_LISTA[col])
                _rotular(ax, t[col], pct)
            ejes_limpios(ax, formato_y=lambda v, _: f"{v:,.0f}%")
        ax.margins(y=0.18)
        ax.axvline(CORTE_ANIO, color="#8c8b87", linestyle=":", linewidth=1)
        ax.set_title(PANELES[p], loc="left", fontsize=10)
        if p != "kg":
            ax.legend(frameon=False, fontsize=8, loc="upper left", bbox_to_anchor=(1.0, 1.0))
    ejes[-1].set_xticks(X_MES, ETIQ_MES, rotation=90)
    fig.suptitle(f"{CORTO[soc]}: evolución mes a mes", x=0.01, ha="left", fontsize=12)
    plt.tight_layout()
    return fig


for vista, soc in VISTAS.items():
    figura_evolucion(soc)
    plt.show()
    t = indicadores(mensual_de(soc))[[*COMPACTO[:3], PRECIO_LISTA, PRECIO_NETO, *PILAR]]
    t.index = t.index.strftime("%Y-%m")
    t.index.name = "Mes"
    display(estilo_columnas(t, f"{vista}: mes a mes"))

## 3. Rangos de % aplicado al precio de lista

Para cada sociedad: la **gráfica** de los 22 rangos (largo = % de la venta bruta del año; color = motivo del
ajuste; al lado de cada barra, el % y el ajuste general en pesos) y la **tabla** del detalle por rango
(participación en venta bruta y kg, y el ajuste en pesos: general, ajuste de precio de mercado y fecha corta).
Las secciones 4 a 6 repiten la vista para cada canal y grupo de material, con una tabla resumen previa.

In [ ]:
def cubo_de(dim: str, soc, miembro=None) -> pd.DataFrame:
    c = cubo.xs(dim, level="Dimension")
    c = c[c.index.get_level_values("Sociedad") == soc]
    if miembro is not None:
        c = c[c.index.get_level_values("Miembro") == miembro]
    return c


def con_rango(c: pd.DataFrame) -> pd.DataFrame:
    """Solo las ventas con venta bruta positiva (las que tienen rango)."""
    return c[c.index.get_level_values("Rango Total") != _ROTULO_SIN_BRUTA]


def miembros(dim: str, soc) -> list:
    """Miembros con venta bruta en el ultimo año, de mayor a menor (top N si aplica)."""
    b = cubo_de(dim, soc)
    b = b[b.index.get_level_values("Año") == A2]["Venta Bruta"].groupby(level="Miembro").sum()
    b = b[b > 0].sort_values(ascending=False)
    return list(b.index[:TOPE.get(dim, len(b))])


def resumen_dim(dim: str, soc, columnas=COMPACTO) -> pd.DataFrame:
    """Miembro x (indicador, año)."""
    s = cubo_de(dim, soc).groupby(level=["Miembro", "Año"]).sum()
    t = indicadores(s)[columnas].unstack("Año")
    t.columns = pd.MultiIndex.from_tuples([(i, str(a_)) for i, a_ in t.columns])
    t = t.reindex(columns=pd.MultiIndex.from_product([columnas, [str(A1), str(A2)]]))
    t.index.name = dim
    return t.reindex(miembros(dim, soc))


def tabla_rangos(c: pd.DataFrame) -> pd.DataFrame:
    """Rango x (indicador, año): participacion y ajustes en pesos (general, ajuste, fecha corta)."""
    r = con_rango(c).groupby(level=["Año", "Rango Total"]).sum()
    out = {}
    for a_ in ANIOS:
        x = r[r.index.get_level_values("Año") == a_].droplevel("Año").reindex(_ROTULOS).fillna(0)
        aj, fc = x["Descuento Ajuste Precio Mercado"], x["Descuento Rotacion / Fecha Corta"]
        out[("% venta bruta", str(a_))] = x["Venta Bruta"] / x["Venta Bruta"].sum() * 100
        out[("% venta kg", str(a_))] = x["Venta Kg"] / x["Venta Kg"].sum() * 100
        out[(P_GEN, str(a_))] = (aj + fc) / MILLON
        out[(P_AJ, str(a_))] = aj / MILLON
        out[(P_FC, str(a_))] = fc / MILLON
    t = pd.DataFrame(out)
    orden = ["% venta bruta", "% venta kg", P_GEN, P_AJ, P_FC]
    t = t[pd.MultiIndex.from_product([orden, [str(A1), str(A2)]])]
    t.index = TRAMOS
    t.index.name = "Rango % aplicado al precio de lista general"
    return t


def rango_principal(c: pd.DataFrame, anio: int):
    """Rango con mas venta bruta entre las ventas CON ajuste (sin el rango 100% o mas). None si no hay venta."""
    r = con_rango(c)
    r = r[(r.index.get_level_values("Año") == anio)
          & (r.index.get_level_values("Rango Total") != _ROTULOS[0])]["Venta Bruta"].groupby(level="Rango Total").sum()
    r = r[r > 0]
    return r.idxmax()[4:] if len(r) else None


def frase_rango(c: pd.DataFrame) -> str:
    """'la venta con ajuste pasó del rango X al Y' / 'sigue concentrada en X'."""
    r1, r2 = rango_principal(c, A1), rango_principal(c, A2)
    if r2 is None:
        return f"sin venta con ajuste en {A2}"
    if r1 is None:
        return f"la venta con ajuste se concentra en “{r2}” (sin venta en {A1})"
    if r1 == r2:
        return f"la venta con ajuste sigue concentrada en “{r2}”"
    return f"la venta con ajuste pasó del rango “{r1}” al “{r2}”"


def figura_rangos(c: pd.DataFrame, titulo: str, figsize=(13, 7.4)):
    """22 rangos (filas) x motivo, un panel por año. Largo = % de la venta bruta del año; al lado, % y ajuste general."""
    cr = con_rango(c)
    r = cr["Venta Bruta"]
    por_motivo = r.groupby(level=["Año", "Rango Total", "Motivo"]).sum()
    dcto = cr["Descuento Ajuste Precio Mercado"] + cr["Descuento Rotacion / Fecha Corta"]
    por_rango = dcto.groupby(level=["Año", "Rango Total"]).sum()
    total_anio = r.groupby(level="Año").sum()
    fig, ejes = plt.subplots(1, len(ANIOS), figsize=figsize, sharey=True, sharex=True)
    tope = 0
    for ax, anio in zip(ejes, ANIOS):
        if anio not in total_anio.index:
            ax.set_title(f"{anio}: sin venta", loc="left", fontsize=11)
            continue
        t = (por_motivo[por_motivo.index.get_level_values("Año") == anio].droplevel("Año").unstack("Motivo")
             .reindex(_ROTULOS).fillna(0) / total_anio[anio] * 100)
        pesos_ = por_rango[por_rango.index.get_level_values("Año") == anio].droplevel("Año").reindex(_ROTULOS).fillna(0)
        base = pd.Series(0.0, index=t.index)
        for m, color in MOTIVOS.items():
            v = t[m] if m in t else pd.Series(0.0, index=t.index)
            ax.barh(TRAMOS, v, left=base, color=color, edgecolor="white", linewidth=1, height=0.75, label=m)
            base += v
        for i, (total_, plata) in enumerate(zip(base, pesos_)):
            if total_ >= 0.1:
                ax.annotate(f"{pct(total_)} · ajuste {pesos(plata)}", (total_, i), xytext=(3, 0),
                            textcoords="offset points", va="center", fontsize=6.5, color="#52514e")
        tope = max(tope, base.max())
        ax.set_title(str(anio), loc="left", fontsize=11)
        ejes_limpios(ax, formato_x=lambda v, _: f"{v:,.0f}%")
    for ax in ejes:
        ax.set_xlim(0, tope * 2.1)  # espacio para la etiqueta del ajuste
    ejes[0].invert_yaxis()
    ejes[-1].legend(loc="lower right", frameon=False, fontsize=8)
    fig.suptitle(titulo, x=0.01, ha="left", fontsize=11)
    plt.tight_layout()
    return fig


def mostrar_dimension(dim: str):
    """Por sociedad: tabla resumen de la dimension (salvo el total) y la grafica + tabla de rangos de cada miembro."""
    for vista, soc in VISTAS.items():
        display(Markdown(f"---\n#### {dim} — {vista}"))
        if dim != "Total":
            res = resumen_dim(dim, soc)
            est = estilo_columnas(res, f"{dim} — {vista}: venta bruta, kg y % aplicado al precio de lista, "
                                       f"{PERIODO} {A1} vs {A2}")
            if dim in CON_CALOR:   # una escala por ajuste, compartida entre los años
                for ind_ in [i for i in res.columns.unique(0) if i.startswith("% Ajuste")]:
                    est = est.background_gradient(cmap="Oranges", low=0.05, high=0.35, axis=None,
                                                  subset=[col for col in res.columns if col[0] == ind_])
            display(est)
        for m in miembros(dim, soc):
            c = cubo_de(dim, soc, m)
            nombre = vista if dim == "Total" else f"{m} — {vista}"
            fig = figura_rangos(c, f"{nombre}: % de la venta bruta por rango de % aplicado al precio de lista general "
                                   f"y motivo, {PERIODO} {A1} vs {A2}")
            lado_a_lado(fig, estilo_columnas(tabla_rangos(c), f"{nombre}: detalle por rango"))


mostrar_dimension("Total")

## 4. Rangos de % de lista precio – Canal de distribución

In [ ]:
mostrar_dimension("Canal de distribución")

## 5. Rangos de % de lista precio – Canal de venta

In [ ]:
mostrar_dimension("Canal de venta")

## 6. Rangos de % de lista precio – Grupo de material 6

Los `TOP_RANGOS` grupos de más venta bruta.

In [ ]:
mostrar_dimension("Grupo de material 6")

## 7. Canal de venta y canal de distribución

Cada canal de venta (tradicional, institucional, modernos, amas de casa) abierto en **Mayoreo y Puntos de Venta**,
con el **Total** al lado: en kg, venta y ajustes en pesos, **Mayoreo + Puntos de Venta = Total**; los precios por
kilo y los % se recalculan desde esas sumas. Acumulado enero a septiembre, 2025 vs 2026, con la variación del total.

In [ ]:
FILAS_CRUCE = [VENTA_KG, VENTA_BRUTA, VENTA_NETA, P_GEN, P_AJ, P_FC, PRECIO_LISTA, PRECIO_NETO,
               L_GEN, D_GEN, L_AJ, D_AJ, L_FC, D_FC]
CANALES_DIST = sorted(cruce.index.unique("Canal de Distribucion"))


def desglose_canal(dim: str, soc, miembro) -> pd.DataFrame:
    """Indicador x (año, canal de distribucion): Mayoreo, Puntos de Venta y Total; mas la variación del total."""
    c = cruce.xs(dim, level="Dimension")
    c = c[(c.index.get_level_values("Sociedad") == soc) & (c.index.get_level_values("Miembro") == miembro)]
    por_canal = c.groupby(level=["Canal de Distribucion", "Año"]).sum()
    total = pd.concat({"Total": c.groupby(level="Año").sum()}, names=["Canal de Distribucion"])
    canales = [x for x in CANALES_DIST if x in por_canal.index.get_level_values(0)] + ["Total"]
    ind = indicadores(pd.concat([por_canal, total]))[FILAS_CRUCE]
    ind = ind.reindex(pd.MultiIndex.from_product([canales, ANIOS], names=["Canal de Distribucion", "Año"]))
    orden = [(canal, a_) for a_ in ANIOS for canal in canales]
    t = ind.T[orden]
    t.columns = pd.MultiIndex.from_tuples([(str(a_), canal) for canal, a_ in orden])
    es_pct = t.index.str.startswith("%")
    tot1, tot2 = t[(str(A1), "Total")], t[(str(A2), "Total")]
    t[(f"Variación {A2} vs {A1}", "Total")] = ((tot2 / tot1 - 1) * 100).where(~es_pct, tot2 - tot1)
    t.index.name = miembro
    return t


# control: el total del cruce cuadra con el de los rangos
for dim in CRUCES:
    for soc in VISTAS.values():
        a_ = cruce.xs(dim, level="Dimension")
        a_ = a_[a_.index.get_level_values("Sociedad") == soc]["Venta Bruta"].sum()
        b_ = cubo_de(dim, soc)["Venta Bruta"].sum()
        assert abs(a_ - b_) < 1, f"El cruce no cuadra en {dim} / {soc}: {a_:,.0f} vs {b_:,.0f}"

for vista, soc in VISTAS.items():
    for dim in CRUCES:
        display(Markdown(f"---\n#### {dim} por canal de distribución — {vista}"))
        for m in miembros(dim, soc):
            display(estilo_filas(desglose_canal(dim, soc, m),
                                 f"{m} — {vista}: Mayoreo + Puntos de Venta = Total, {PERIODO} {A1} vs {A2}"))

## 8. Concentración del ajuste – Distribuidoras

Acumulado enero a septiembre. Primero, las distribuidoras ordenadas por **ajuste general** del último año, con su %
acumulado, la venta y los tres % aplicados al precio de lista; después, los rangos de las `TOP_RANGOS` de más venta
bruta.

In [ ]:
def pareto(s: pd.DataFrame, nombre: str) -> pd.DataFrame:
    """s: sumas por (miembro, año). Ordena por ajuste general del ultimo año y acumula."""
    ind = indicadores(s)
    w2 = ind[ind.index.get_level_values("Año") == A2].droplevel("Año").sort_values(P_GEN, ascending=False)
    w1 = ind[ind.index.get_level_values("Año") == A1].droplevel("Año").reindex(w2.index)
    a1, a2 = str(A1), str(A2)
    out = pd.DataFrame({
        (VENTA_KG, a1): w1[VENTA_KG], (VENTA_KG, a2): w2[VENTA_KG],
        (VENTA_BRUTA, a2): w2[VENTA_BRUTA], (P_GEN, a1): w1[P_GEN], (P_GEN, a2): w2[P_GEN],
        (VENTA_NETA, a2): w2[VENTA_NETA],
        ("% acumulado ajuste general", a2): w2[P_GEN].clip(lower=0).cumsum() / w2[P_GEN].clip(lower=0).sum() * 100,
        (L_GEN, a1): w1[L_GEN], (L_GEN, a2): w2[L_GEN], (D_GEN, a2): w2[D_GEN],
        (L_AJ, a2): w2[L_AJ], (D_AJ, a2): w2[D_AJ],
        (L_FC, a1): w1[L_FC], (L_FC, a2): w2[L_FC], (D_FC, a1): w1[D_FC], (D_FC, a2): w2[D_FC],
    })
    out.index.name = nombre
    return out


def cuantos_para(s: pd.Series, porcentaje: float) -> int:
    """Cuantos miembros (de mayor a menor) suman el porcentaje% del total."""
    s = s[s > 0].sort_values(ascending=False)
    return int((s.cumsum() / s.sum() * 100 < porcentaje).sum() + 1) if len(s) else 0


def distribuidoras(soc) -> pd.DataFrame:
    return pareto(cubo_de("Distribuidora", soc).groupby(level=["Miembro", "Año"]).sum(), "Distribuidora")


def mitad_distribuidoras(soc) -> list:
    """Distribuidoras que juntas suman la mitad del ajuste general del ultimo año."""
    t = distribuidoras(soc)
    return list(t.index[: int((t[("% acumulado ajuste general", str(A2))] < 50).sum()) + 1])


for vista, soc in VISTAS.items():
    display(Markdown(f"---\n#### Distribuidoras — {vista} ({PERIODO} {A1} vs {A2})"))
    display(estilo_columnas(distribuidoras(soc), f"Distribuidoras — {vista}: concentración del ajuste general")
            .bar(subset=[("% acumulado ajuste general", str(A2))], color="#cfe0f5", vmin=0, vmax=100))
mostrar_dimension("Distribuidora")

## 9. Concentración del ajuste – Clientes Mayoreo / Puntos de Venta

> **Mes analizado: el de `MES_CLIENTES` (A.1)**, contra el mismo mes del año anterior. Para revisar otro mes,
> cambie ese parámetro y vuelva a ejecutar desde A.6.

Por separado para Mayoreo y Puntos de Venta: la concentración del ajuste entre clientes y los `TOP_CLIENTES`
que más compraron con ajuste.

In [ ]:
CANALES = sorted(clientes.index.unique("Canal de Distribucion"))


def clientes_de(soc, canal=None) -> pd.DataFrame:
    k = clientes[clientes.index.get_level_values("Sociedad") == soc]
    if canal:
        k = k[k.index.get_level_values("Canal de Distribucion") == canal]
    return k.groupby(level=["Codigo SAP Cliente", "Año"]).sum()


def top_clientes(soc, canal=None, n=TOP_CLIENTES, solo_ultimo=True) -> pd.DataFrame:
    """Top n por ajuste general del mes de clientes.

    solo_ultimo=True (tabla del informe): solo el ultimo año, el canal de venta y los ajustes (general,
    ajuste y fecha corta), cada uno con su valor en pesos completos al lado de su %.
    solo_ultimo=False: el pareto completo con el año anterior (lo usan los hallazgos).
    """
    if not solo_ultimo:
        t = pareto(clientes_de(soc, canal), "Cliente").head(n)
    else:
        k = clientes_de(soc, canal)
        ind = indicadores(k[k.index.get_level_values("Año") == A2].droplevel("Año"))
        ind = ind.sort_values(P_GEN, ascending=False).head(n)
        t = pd.DataFrame({
            "Canal de venta": [canal_venta_cliente.get(cod, "-") for cod in ind.index],
            VENTA_KG: ind[VENTA_KG],
            VENTA_BRUTA: ind[VENTA_BRUTA] * MILLON,
            P_GEN: ind[P_GEN] * MILLON,
            D_GEN: ind[D_GEN],
            P_AJ: ind[P_AJ] * MILLON,
            D_AJ: ind[D_AJ],
            P_FC: ind[P_FC] * MILLON,
            D_FC: ind[D_FC],
        }, index=ind.index)
    t.index = [f"{cod} - {nombre_cliente.get(cod, '')}" for cod in t.index]
    t.index.name = "Cliente"
    return t


def concentracion(soc, canal=None) -> pd.DataFrame:
    k = clientes_de(soc, canal)
    filas = {}
    for anio in ANIOS:
        x = k[k.index.get_level_values("Año") == anio].droplevel("Año")
        gen = x["Descuento Ajuste Precio Mercado"] + x["Descuento Rotacion / Fecha Corta"]
        filas[str(anio)] = {
            "Clientes con venta": int((x["Venta Bruta"] > 0).sum()),
            "Clientes con ajuste": int((gen > 0).sum()),
            "Clientes con el 50% del ajuste general": cuantos_para(gen, 50),
            "Clientes con el 80% del ajuste general": cuantos_para(gen, 80),
            "Clientes con fecha corta": int((x["Descuento Rotacion / Fecha Corta"] > 0).sum()),
        }
    return pd.DataFrame(filas)


COLS_TOP = ["Canal de venta", "Distribuidora", VENTA_KG, VENTA_BRUTA, P_GEN, D_GEN, P_AJ, D_AJ, P_FC, D_FC]


def principal_top(k: pd.DataFrame, nivel: str) -> pd.Series:
    """Valor del nivel (canal de venta / distribuidora) con mas venta bruta para cada cliente."""
    x = k["Venta Bruta"].groupby(level=["Codigo SAP Cliente", nivel]).sum().reset_index()
    x = x.sort_values("Venta Bruta").drop_duplicates("Codigo SAP Cliente", keep="last")
    return x.set_index("Codigo SAP Cliente")[nivel]


def top_de(k: pd.DataFrame, n=TOP_CLIENTES):
    """Top n clientes por ajuste general ($). Devuelve (tabla, % del ajuste general que suman, total, clientes con ajuste)."""
    g = k.groupby(level="Codigo SAP Cliente")[MEDIDAS].sum()
    gen = g["Descuento Ajuste Precio Mercado"] + g["Descuento Rotacion / Fecha Corta"]
    g, gen = g[gen > 0], gen[gen > 0]
    if g.empty:
        return None, 0, 0, 0
    total, n_cli = gen.sum(), len(g)
    orden = gen.sort_values(ascending=False).head(n).index
    g, gen = g.loc[orden], gen.loc[orden]
    ind = indicadores(g)
    canal, dist = principal_top(k, "Canal de Venta"), principal_top(k, "Nombre Distribuidora")
    t = pd.DataFrame({
        "Canal de venta": [canal.get(c_, "-") for c_ in g.index],
        "Distribuidora": [str(dist.get(c_, "-")).title() for c_ in g.index],
        VENTA_KG: ind[VENTA_KG],
        VENTA_BRUTA: g["Venta Bruta"],
        P_GEN: gen,
        D_GEN: ind[D_GEN],
        P_AJ: g["Descuento Ajuste Precio Mercado"],
        D_AJ: ind[D_AJ],
        P_FC: g["Descuento Rotacion / Fecha Corta"],
        D_FC: ind[D_FC],
    }, index=g.index)[COLS_TOP]
    t.index = [f"{cod} - {nombre_cliente.get(cod, '')}" for cod in t.index]
    t.index.name = "Cliente"
    return t, gen.sum() / total * 100, total, n_cli


for vista, soc in VISTAS.items():
    for canal in CANALES:
        display(Markdown(f"---\n#### Clientes {canal} — {vista} ({MES_TXT} {A2} vs {MES_TXT} {A1})"))
        display(concentracion(soc, canal).style.format(lambda v: num(v, 0))
                .set_caption(f"Concentración de clientes {canal} — {vista} — {MES_TXT}"))
        tc = top_clientes(soc, canal)
        display(estilo_columnas(tc, f"Top {TOP_CLIENTES} clientes {canal} — {vista}: los que más compraron con "
                                    f"ajuste en {MES_TXT} {A2}")
                .background_gradient(cmap="Oranges", subset=[D_GEN, D_AJ, D_FC], low=0.05, high=0.35))

## 10. Avance del mes en curso

El mes anterior cerrado contra el mes en curso hasta `CORTE_D` (por defecto ayer), con las mismas tablas y gráficas de
las secciones 2 a 9: panorama (con la venta por día), evolución día a día, rangos por dimensión, canal de venta por
canal de distribución, distribuidoras y clientes. `modo_corte()` cambia, solo mientras dura el `with`, los periodos y
las tablas que leen las funciones de B y C (al salir todo queda como estaba); el informe de avance (C.6) usa las
mismas láminas del informe general. Los kg y pesos del mes en curso son parciales: compare los %, los precios por kg
y la venta por día.

In [ ]:
DIAS_EJE = pd.period_range(INICIO_ANT, FECHA_MAX_D, freq="D")
COLS_FOCO = [VENTA_KG, P_GEN, PRECIO_NETO, L_GEN, D_AJ, D_FC]
KG_DIA, BRUTA_DIA = "Venta kg por día", "Venta bruta por día ($)"


def diario_de(soc) -> pd.DataFrame:
    """Sumas por dia de la sociedad (en el modo de corte reemplaza a mensual_de)."""
    return diario_soc[diario_soc.index.get_level_values("Sociedad") == soc].groupby(level="Dia").sum()


def dias_con_venta(soc) -> pd.Series:
    """Dias con venta bruta en cada periodo."""
    dd = diario_de(soc)
    dd = dd[dd["Venta Bruta"] > 0]
    return pd.Series(dd.index.to_timestamp() >= pd.Timestamp(INICIO_ACT), index=dd.index).map(
        {False: ETQ_ANT, True: ETQ_ACT}).value_counts().reindex(PERIODOS_D, fill_value=0)


def panorama_corte(soc) -> pd.DataFrame:
    """La tabla del panorama (A1 = mes anterior, A2 = mes en curso) mas la venta por dia con venta."""
    s = cubo_d.xs("Total", level="Dimension")
    s = s[s.index.get_level_values("Sociedad") == soc].groupby(level="Año").sum().reindex(PERIODOS_D)
    ind = indicadores(s)
    dias = dias_con_venta(soc)
    ind[KG_DIA] = ind[VENTA_KG] / dias
    ind[BRUTA_DIA] = ind[VENTA_BRUTA] / dias
    return comparar(ind)


CAMBIOS_CORTE = {
    "A1": ETQ_ANT, "A2": ETQ_ACT, "ANIOS": PERIODOS_D, "PERIODO": "mes cerrado vs en curso ·",
    "MES_FIN_TXT": f"{MES_ACT_TXT} al {FECHA_MAX_D.day}", "MES_TXT": ETQ_ACT, "FECHA_MAX": FECHA_MAX_D,
    "cubo": cubo_d, "cruce": cruce_d, "clientes": clientes_d, "nombre_cliente": nombre_cliente_d,
    "canal_venta_cliente": canal_venta_cliente_d,
    "CANALES": sorted(clientes_d.index.unique("Canal de Distribucion")),
    "CANALES_DIST": sorted(cruce_d.index.unique("Canal de Distribucion")),
    # evolucion dia a dia (seccion 2)
    "mensual_de": diario_de, "MESES_EJE": DIAS_EJE, "X_MES": list(range(len(DIAS_EJE))),
    "ETIQ_MES": DIAS_EJE.strftime("%d/%m"), "CORTE_ANIO": sum(DIAS_EJE.month == INICIO_ANT.month) - 0.5,
    "PANELES": {"kg": "Venta kg por día",
                "pesos": "Venta bruta y venta neta por día ($) — la franja es el ajuste",
                "precio": "Precio de lista y precio neto por kg, por día",
                "aplicado": "% aplicado al precio de lista por día (100% − % de ajuste)"},
    # rotulos del informe general (C.2) que llevan los años
    "ANIOS_CONC": {L_GEN: (ETQ_ANT, ETQ_ACT), D_FC: (ETQ_ANT, ETQ_ACT), L_FC: (ETQ_ANT, ETQ_ACT)},
    "PANORAMA": None,   # se calcula al entrar
}


@contextmanager
def modo_corte():
    """Mientras dura el `with`, las funciones de B y C comparan el mes anterior cerrado con el mes en curso; al salir
    (aunque haya error) todo queda como estaba. Los rotulos de las secciones (C.2) se toman si ya existen."""
    g = globals()
    cambios = {**CAMBIOS_CORTE, "SECCIONES": {**g.get("SECCIONES", {}), 2: f"Panorama {ETQ_ANT} vs {ETQ_ACT}"}}
    antes = {k: g[k] for k in cambios if k in g}
    g.update(cambios)
    try:
        g["PANORAMA"] = {vista: panorama_corte(soc) for vista, soc in VISTAS.items()}
        yield
    finally:
        for k in cambios:
            if k in antes:
                g[k] = antes[k]
            else:
                g.pop(k, None)


with modo_corte():
    display(Markdown(f"#### Panorama {ETQ_ANT} vs {ETQ_ACT}"))
    display(estilo_filas(pd.concat(PANORAMA, axis=1),
                         f"Panorama {ETQ_ANT} vs {ETQ_ACT}: venta bruta, kg y % aplicado al precio de lista"))
    for vista, soc in VISTAS.items():
        fig = figura_evolucion(soc)
        fig.suptitle(f"{vista}: evolución día a día ({ETQ_ANT} y {ETQ_ACT})", x=0.01, ha="left", fontsize=12)
        plt.show()
    # rangos: total, canales, grupos de material 6 y distribuidoras (como las secciones 3 a 6 y 8)
    for dim in DIMENSIONES:
        mostrar_dimension(dim)
    for vista, soc in VISTAS.items():
        display(Markdown(f"---\n#### Canal de venta por canal de distribución — {vista}"))
        for m in miembros("Canal de venta", soc):
            display(estilo_filas(desglose_canal("Canal de venta", soc, m),
                                 f"{m} — {vista}: Mayoreo + Puntos de Venta = Total, {ETQ_ANT} vs {ETQ_ACT}"))
        display(Markdown(f"---\n#### Distribuidoras y clientes — {vista}"))
        display(estilo_columnas(distribuidoras(soc), f"Distribuidoras — {vista}: concentración del ajuste general")
                .bar(subset=[("% acumulado ajuste general", ETQ_ACT)], color="#cfe0f5", vmin=0, vmax=100))
        for canal in CANALES:
            display(concentracion(soc, canal).style.format(lambda v: num(v, 0))
                    .set_caption(f"Concentración de clientes {canal} — {vista}"))
            display(estilo_columnas(top_clientes(soc, canal), f"Top {TOP_CLIENTES} clientes {canal} — {vista}: los que "
                                                              f"más compraron con ajuste en {ETQ_ACT}")
                    .background_gradient(cmap="Oranges", subset=[D_GEN, D_AJ, D_FC], low=0.05, high=0.35))

## 11. Foco Res

Molida-Hamburguesa Res y Res Ref y Cong (`FOCO`), al mismo corte del avance, por región y canal de venta
(`CANALES_VENTA_TOP`). Para cada material: la tabla del canal (Mayoreo + Puntos de Venta = Total), las gráficas de
rangos del canal, de Mayoreo y de Puntos de Venta con su detalle, y el top de distribuidoras y de clientes.
`solo_canal()` deja en `foco_d` solo un canal de venta mientras dura el `with`.

In [ ]:
def foco_de(soc, material, periodo=None) -> pd.DataFrame:
    """Filas de foco_d de un material en la sociedad (y en un periodo, si se da)."""
    f = foco_d[(foco_d.index.get_level_values("Sociedad") == soc)
               & (foco_d.index.get_level_values("Nombre GR6") == material)]
    if periodo is not None:
        f = f[f.index.get_level_values("Año") == periodo]
    return f


def a_columnas(filas: dict, nombre: str) -> pd.DataFrame:
    """{rotulo: indicadores por periodo} -> rotulo x (indicador, periodo) con COLS_FOCO, en el orden del dict."""
    t = pd.concat({k: v[COLS_FOCO].reindex(ANIOS) for k, v in filas.items()}, names=[nombre, "Año"])
    t = t.unstack("Año").reindex(columns=pd.MultiIndex.from_product([COLS_FOCO, ANIOS])).reindex(list(filas))
    t.index.name = nombre
    return t


@contextmanager
def solo_canal(canal_venta):
    """Mientras dura el `with`, foco_d (el vigente: mes o acumulado) solo trae el canal de venta dado."""
    g = globals()
    antes = g["foco_d"]
    g["foco_d"] = antes[antes.index.get_level_values("Canal de Venta") == canal_venta]
    try:
        yield
    finally:
        g["foco_d"] = antes


def canales_dist_foco(soc, m) -> list:
    """Canales de distribucion con venta del material (Mayoreo, Puntos de Venta y luego los demas)."""
    b = foco_de(soc, m)["Venta Bruta"].groupby(level="Canal de Distribucion").sum()
    cds = b[b > 0].index
    return [c for c in ROTULO_CANAL if c in cds] + sorted(c for c in cds if c not in ROTULO_CANAL)


def foco_canal_cd(soc, m) -> pd.DataFrame:
    """Material en el canal de venta vigente: cada canal de distribucion y el total del canal (x indicador, periodo)."""
    f = foco_de(soc, m)
    cd = indicadores(f.groupby(level=["Canal de Distribucion", "Año"]).sum())
    filas = {ROTULO_CANAL.get(c, c): cd.xs(c, level="Canal de Distribucion") for c in canales_dist_foco(soc, m)}
    filas["Total"] = indicadores(f.groupby(level="Año").sum())
    return a_columnas(filas, m)


def foco_distribuidoras(soc, material) -> pd.DataFrame:
    """Pareto de distribuidoras del material, ordenadas por ajuste general del mes en curso."""
    s = foco_de(soc, material).groupby(level=["Nombre Distribuidora", "Año"]).sum()
    return pareto(s.rename_axis(["Miembro", "Año"]), "Distribuidora")


def foco_clientes(soc, material):
    """Top de clientes del material en el ultimo periodo (como C.3), con su canal de distribucion y el % de ajuste
    general del periodo anterior. Devuelve (tabla, % del ajuste general del material que suman, ajuste general,
    clientes con ajuste)."""
    k = foco_de(soc, material, A2)
    t, part, total, n_cli = top_de(k)
    if t is None:
        return None, 0, 0, 0
    codigos = [i.split(" - ", 1)[0] for i in t.index]
    canal = principal_top(k, "Canal de Distribucion")
    t.insert(1, "Canal de distribución", [ROTULO_CANAL.get(canal.get(c_), canal.get(c_, "-")) for c_ in codigos])
    ant = indicadores(foco_de(soc, material, A1).groupby(level="Codigo SAP Cliente")[MEDIDAS].sum())[D_GEN]
    t.insert(t.columns.get_loc(D_GEN) + 1, f"{D_GEN} {A1}", [ant.get(c_, float("nan")) for c_ in codigos])
    return t, part, total, n_cli


def calor_ajustes(t: pd.DataFrame, indice) -> pd.DataFrame:
    """Mapa de calor solo en las columnas de % de ajuste (1 o 2 niveles de encabezado)."""
    es_aj = [(c[0] if isinstance(c, tuple) else c).startswith("% Ajuste") for c in t.columns]
    calor = t[[c for c, s in zip(t.columns, es_aj) if s]].reindex(columns=t.columns)
    calor.index = indice
    return calor


with modo_corte():
    for vista, soc in VISTAS.items():
        for cv in CANALES_VENTA_TOP:
            with solo_canal(cv):
                for m in FOCO:
                    if foco_de(soc, m)["Venta Bruta"].sum() <= 0:
                        continue
                    display(Markdown(f"---\n#### Foco Res: {m} · {cv} — {vista} ({ETQ_ANT} vs {ETQ_ACT})"))
                    t = foco_canal_cd(soc, m)
                    display(estilo_columnas(t, f"{m} · {cv} — {vista}: Mayoreo + Puntos de Venta = Total del canal")
                            .background_gradient(cmap="Oranges", low=0.05, high=0.35, axis=None,
                                                 subset=[c for c in t.columns if c[0].startswith("% Ajuste")]))
                    c = foco_de(soc, m)
                    for cd in [None, *canales_dist_foco(soc, m)]:
                        cc = c if cd is None else c[c.index.get_level_values("Canal de Distribucion") == cd]
                        nombre = f"{m} · {cv}" + (f" · {ROTULO_CANAL.get(cd, cd)}" if cd else "")
                        fig = figura_rangos(cc, f"{nombre} — {vista}: % de la venta bruta por rango de % aplicado al "
                                                f"precio de lista general y motivo, {ETQ_ANT} vs {ETQ_ACT}")
                        lado_a_lado(fig, estilo_columnas(tabla_rangos(cc), f"{nombre}: detalle por rango"))
                    display(estilo_columnas(foco_distribuidoras(soc, m).head(TOP_DISTRIBUIDORAS),
                                            f"{m} · {cv} — {vista}: top {TOP_DISTRIBUIDORAS} distribuidoras por ajuste "
                                            f"general de {ETQ_ACT}"))
                    tc = foco_clientes(soc, m)[0]
                    if tc is not None:
                        display(estilo_columnas(tc, f"{m} · {cv} — {vista}: top {TOP_CLIENTES} clientes por ajuste "
                                                    f"general de {ETQ_ACT}")
                                .background_gradient(cmap="Oranges", low=0.05, high=0.35,
                                                     subset=[c for c in tc.columns if c.startswith("% Ajuste")]))

## 12. Hallazgos

Resumen y hallazgos adicionales por sociedad. Los textos se arman con los números de las secciones anteriores y
son los mismos del PowerPoint.

In [ ]:
def v_(vista, ind, anio):
    return PANORAMA[vista].loc[ind, str(anio)]


def ch_(vista, ind):
    return PANORAMA[vista].loc[ind, "Variación"]


def sube_baja(v, plural=False):
    return ("subieron" if v >= 0 else "bajaron") if plural else ("subió" if v >= 0 else "bajó")


def de_a(vista, ind):
    return f"{formato_de(ind)(v_(vista, ind, A1))} → {formato_de(ind)(v_(vista, ind, A2))}"


def es_especial(nombre) -> bool:
    return any(p in str(nombre).upper() for p in ESPECIALES)


def por_anio(s: pd.DataFrame, filtro) -> pd.DataFrame:
    """Sumas por año de los miembros que cumplen el filtro (s: miembro x año)."""
    m = pd.Series([filtro(x) for x in s.index.get_level_values("Miembro")], index=s.index)
    return s[m].groupby(level="Año").sum().reindex(ANIOS)


def resumen_ejecutivo(vista) -> list:
    soc = VISTAS[vista]
    n50 = {canal: concentracion(soc, canal).loc["Clientes con el 50% del ajuste general", str(A2)] for canal in CANALES}
    mitad = mitad_distribuidoras(soc)
    fr = frase_rango(cubo_de("Total", soc))
    return [
        ("Precio. ", f"La lista por kilo {sube_baja(ch_(vista, PRECIO_LISTA))} {num(abs(ch_(vista, PRECIO_LISTA)))}% "
                     f"y el neto por kilo {sube_baja(ch_(vista, PRECIO_NETO))} {num(abs(ch_(vista, PRECIO_NETO)))}% "
                     f"({PERIODO}). % aplicado al precio de lista general: {de_a(vista, L_GEN)}."),
        ("Ajuste de precio de mercado. ", f"% aplicado al precio de lista ajuste de precio de mercado: {de_a(vista, L_AJ)}; el ajuste de precio de mercado "
                                f"llegó a {pesos(v_(vista, P_AJ, A2))} ({variacion(ch_(vista, P_AJ), False)})."),
        ("Fecha corta. ", f"% aplicado al precio de lista fecha corta: {de_a(vista, L_FC)}; el ajuste de fecha corta llegó a "
                          f"{pesos(v_(vista, P_FC, A2))} ({variacion(ch_(vista, P_FC), False)})."),
        ("Volumen. ", f"La venta kg {sube_baja(ch_(vista, VENTA_KG))} {num(abs(ch_(vista, VENTA_KG)))}% y la venta bruta "
                      f"{sube_baja(ch_(vista, VENTA_BRUTA))} {num(abs(ch_(vista, VENTA_BRUTA)))}%: el crecimiento en "
                      "pesos viene del precio de lista, no del volumen."),
        ("Rangos. ", f"{fr[0].upper()}{fr[1:]} de % aplicado al precio de lista general."),
        ("Concentración. ", f"{len(mitad)} distribuidoras concentran la mitad del ajuste general {PERIODO} "
                            f"({', '.join(n_.title() for n_ in mitad)}); en {MES_TXT} de {A2}, "
                            + " y ".join(f"{num(n_, 0)} clientes de {c_}" for c_, n_ in n50.items())
                            + " concentran la mitad del ajuste general de su canal."),
    ]


def hallazgos_adicionales(vista) -> list:
    soc = VISTAS[vista]
    h = []
    # 1. canales especiales (siempre 100% de la lista) y la venta comercial sin ellos
    s = cubo_de("Distribuidora", soc).groupby(level=["Miembro", "Año"]).sum()
    esp, com = por_anio(s, es_especial), por_anio(s, lambda x: not es_especial(x))
    i_esp, i_com = indicadores(esp), indicadores(com)
    nombres_esp = [m.title() for m in s.index.unique("Miembro") if es_especial(m)]
    part_esp = esp.loc[A2, "Venta Kg"] / (esp.loc[A2, "Venta Kg"] + com.loc[A2, "Venta Kg"]) * 100
    h.append(("Canales que siempre facturan a lista completa. ",
              f"{', '.join(nombres_esp)} son el {pct(part_esp)} de la venta kg de {A2} y se facturan al "
              f"{pct(i_esp.loc[A2, L_GEN])} de la lista. Sin ellos, el % aplicado al precio de lista general pasa de "
              f"{pct(i_com.loc[A1, L_GEN])} a {pct(i_com.loc[A2, L_GEN])} y el precio neto por kg de "
              f"{pesos(i_com.loc[A1, PRECIO_NETO])} a {pesos(i_com.loc[A2, PRECIO_NETO])}."))
    # 2. volumen comercial y donde se perdio
    kg = s["Venta Kg"].unstack("Año").reindex(columns=ANIOS).fillna(0)
    kg = kg[[not es_especial(m) for m in kg.index]]
    caidas = (kg[A2] - kg[A1]).sort_values().head(3)
    dif_com = com.loc[A2, "Venta Kg"] - com.loc[A1, "Venta Kg"]
    dif_esp = esp.loc[A2, "Venta Kg"] - esp.loc[A1, "Venta Kg"]
    h.append(("Volumen. ", f"La venta kg comercial (sin esos canales) {sube_baja(dif_com)} {num(abs(dif_com), 0)} kg "
                           f"({variacion((com.loc[A2, 'Venta Kg'] / com.loc[A1, 'Venta Kg'] - 1) * 100, False)}); "
                           "las mayores caídas: " + ", ".join(f"{m_.title()} ({num(v, 0)} kg)" for m_, v in caidas.items())
                           + f". Los canales de lista completa {sube_baja(dif_esp, True)} {num(abs(dif_esp), 0)} kg."))
    # 3. evolucion: punto mas bajo y ultimo mes
    m = indicadores(mensual_de(soc))
    m2 = m[m.index.year == A2]
    od = mensual_de(soc)
    od = od[(od.index.year == A1) & (od.index.month >= 10)].sum()
    i_od = indicadores(od.to_frame().T).iloc[0] if od["Venta Bruta"] else None
    h.append(("Evolución. ", f"El % aplicado al precio de lista general tocó su punto más bajo en "
                             f"{MESES_ES[m2[L_GEN].idxmin().month - 1]} de {A2} ({pct(m2[L_GEN].min())}); en {MES_FIN_TXT} "
                             f"está en {pct(m2[L_GEN].iloc[-1])}"
                             + (f". En octubre–diciembre de {A1} estaba en {pct(i_od[L_GEN])}." if i_od is not None else ".")))
    # 4. distribuidoras: mayor caida y menor % aplicado fecha corta (sin canales especiales)
    rd = resumen_dim("Distribuidora", soc, [VENTA_KG, L_GEN, L_FC])
    rd = rd[[not es_especial(x) for x in rd.index]]
    caida = (rd[(L_GEN, str(A2))] - rd[(L_GEN, str(A1))]).idxmin()
    peor_fc = rd[(L_FC, str(A2))].idxmin()
    texto_fc = (f"{'y también' if peor_fc == caida else f'; {peor_fc.title()}'} tiene el menor % aplicado al precio "
                f"de lista fecha corta ({pct(rd.loc[peor_fc, (L_FC, str(A1))])} → {pct(rd.loc[peor_fc, (L_FC, str(A2))])})")
    h.append(("Distribuidoras. ", f"{caida.title()} tuvo la mayor caída del % aplicado al precio de lista general "
                                  f"({pct(rd.loc[caida, (L_GEN, str(A1))])} → {pct(rd.loc[caida, (L_GEN, str(A2))])})"
                                  f"{' ' if peor_fc == caida else ''}{texto_fc}."))
    # 5. canal de venta con mayor caida y que ajuste la explica
    cv = resumen_dim("Canal de venta", soc, [L_GEN, L_AJ, L_FC])
    peor = (cv[(L_GEN, str(A2))] - cv[(L_GEN, str(A1))]).idxmin()
    d_aj = cv.loc[peor, (L_AJ, str(A2))] - cv.loc[peor, (L_AJ, str(A1))]
    d_fc = cv.loc[peor, (L_FC, str(A2))] - cv.loc[peor, (L_FC, str(A1))]
    h.append(("Canales de venta. ", f"{peor} tuvo la mayor caída del % aplicado al precio de lista general "
                                    f"({pct(cv.loc[peor, (L_GEN, str(A1))])} → {pct(cv.loc[peor, (L_GEN, str(A2))])}), "
                                    f"sobre todo por {'el ajuste de precio de mercado' if d_aj < d_fc else 'la fecha corta'} "
                                    f"(ajuste {variacion(d_aj, True)}; fecha corta {variacion(d_fc, True)})."))
    # 6. venta a menos del 50% de la lista
    r = con_rango(cubo_de("Total", soc))
    lim = r.index.get_level_values("Rango Total").map(LIMITE)
    kg_tot = r["Venta Kg"].groupby(level="Año").sum()
    bajo = r[lim < 50]
    kg_bajo = bajo["Venta Kg"].groupby(level="Año").sum().reindex(ANIOS, fill_value=0)
    fc_bajo = bajo[bajo.index.get_level_values("Motivo") == "Con fecha corta"]["Venta Kg"].groupby(level="Año").sum()
    h.append(("Venta a menos del 50% de la lista. ", f"Pasó de {pct(kg_bajo[A1] / kg_tot[A1] * 100)} a "
                                                     f"{pct(kg_bajo[A2] / kg_tot[A2] * 100)} de la venta kg; el "
                                                     f"{pct(fc_bajo.get(A2, 0) / kg_bajo[A2] * 100)} tuvo fecha corta en {A2}."))
    # 7. clientes de mayoreo del mes de clientes
    if "Mayoreo" in CANALES:
        k = clientes_de(soc, "Mayoreo")
        x1 = k[k.index.get_level_values("Año") == A1].droplevel("Año")
        x2 = k[k.index.get_level_values("Año") == A2].droplevel("Año")
        nuevos, perdidos = x2.index.difference(x1.index), x1.index.difference(x2.index)
        cc = concentracion(soc, "Mayoreo")
        h.append((f"Clientes de mayoreo en {MES_TXT}. ",
                  f"Con fecha corta: {num(cc.loc['Clientes con fecha corta', str(A1)], 0)} → "
                  f"{num(cc.loc['Clientes con fecha corta', str(A2)], 0)}. Clientes nuevos frente a {MES_TXT} de {A1}: "
                  f"{num(len(nuevos), 0)} ({pct(x2.loc[nuevos, 'Venta Kg'].sum() / x2['Venta Kg'].sum() * 100)} de la venta kg); "
                  f"clientes que no volvieron: {num(len(perdidos), 0)} "
                  f"({pct(x1.loc[perdidos, 'Venta Kg'].sum() / x1['Venta Kg'].sum() * 100)} de la venta kg de {A1})."))
        top = top_clientes(soc, "Mayoreo", solo_ultimo=False)
        cambiaron = top[(top[(D_FC, str(A2))] > 30) & (top[(D_FC, str(A1))].fillna(0) < 5)]
        if len(cambiaron):
            h.append(("Clientes que cambiaron de patrón. ",
                      f"{len(cambiaron)} de los {TOP_CLIENTES} clientes de mayoreo con más ajuste en {MES_TXT} "
                      f"{'pasó' if len(cambiaron) == 1 else 'pasaron'} de menos de 5% a más de 30% de ajuste de fecha corta (p. ej. "
                      + ", ".join(i.split(" - ", 1)[-1].title() for i in cambiaron.index[:3])
                      + "): candidatos a revisar autorizaciones."))
    h.append(("Calidad de datos. ", "Las devoluciones se registran a precio de lista, sin ajuste, y bajan el rango de "
                                    "las ventas con las que se agrupan: afectan la distribución por rangos (sobre todo los "
                                    "más bajos), no los totales. El ajuste foco casi no se usa."))
    return h


def como_markdown(items):
    return "\n".join(f"- **{t.strip()}** {c}" for t, c in items)


for vista in VISTAS:
    display(Markdown(f"#### {vista} — resumen\n" + como_markdown(resumen_ejecutivo(vista))))
    display(Markdown(f"#### {vista} — hallazgos adicionales\n" + como_markdown(hallazgos_adicionales(vista))))

# C. Salidas

Requieren las partes A y B: aquí solo se generan los archivos, en `SALIDA`:

| Sección | Archivo |
|---|---|
| **Acumulado** | |
| C.2 Informe general | `Informe_Ajustes_<región>_<año>_vs_<año>.pptx` |
| C.3 Top de clientes | `Top_Clientes_<región>_<Distribuidoras / Canales_de_venta>_<Mes>_<año>.pptx` (4 archivos) |
| C.4 Un cliente | `Informe_Cliente_<código>_<año>_vs_<año>.pptx` |
| C.5 Panorama Región Oriente vs Región Occidente | `Panorama_<año>_vs_<año>.xlsx` (único Excel) |
| **Mes en curso (al corte)** | |
| C.6 Informe de avance | `Informe_Avance_<región>_<mes anterior>_vs_<mes en curso>_al_<día>.pptx` |
| C.7 Foco Res | `Foco_Res_<región>_<canal de venta>_<mes anterior>_vs_<mes en curso>_al_<día>.pptx` (8 archivos) |

## C.1 Funciones comunes de PowerPoint

In [ ]:
from pptx import Presentation
from pptx.dml.color import RGBColor
from pptx.enum.text import MSO_ANCHOR, PP_ALIGN
from pptx.util import Inches, Pt


def rgb(h: str) -> RGBColor:
    return RGBColor.from_string(h.lstrip("#").upper())


TINTA, TINTA_2, TINTA_3 = rgb("#0b0b0b"), rgb("#52514e"), rgb("#8c8b87")
FONDO_ENC, BLANCO, ACENTO = rgb("#f0efeb"), rgb("#ffffff"), rgb("#2a78d6")
ANCHO, ALTO, MARGEN = Inches(13.333), Inches(7.5), Inches(0.5)
UTIL_W = ANCHO - 2 * MARGEN
CUERPO_Y = Inches(1.6)
CUERPO_H = ALTO - CUERPO_Y - Inches(0.55)
prs = None   # presentacion en construccion


def nueva_presentacion():
    global prs
    prs = Presentation()
    prs.slide_width, prs.slide_height = ANCHO, ALTO


def guardar(nombre: str):
    archivo_ = SALIDA / nombre
    try:
        prs.save(archivo_)
        print(f"{archivo_.name}: {len(prs.slides)} láminas")
    except PermissionError:
        print(f"No se pudo guardar {archivo_.name}: ciérrelo en PowerPoint y vuelva a correr esta celda.")


def caja_texto(d, x, y, w, h, parrafos, tam=14, color=TINTA, negrita=False, alinear=PP_ALIGN.LEFT, espacio=0):
    """parrafos: texto o lista; cada parrafo es texto o lista de (texto, negrita)."""
    caja = d.shapes.add_textbox(x, y, w, h)
    tf = caja.text_frame
    tf.word_wrap = True
    for i, par in enumerate([parrafos] if isinstance(parrafos, str) else parrafos):
        p = tf.paragraphs[0] if i == 0 else tf.add_paragraph()
        p.alignment = alinear
        p.space_after = Pt(espacio)
        for trozo, neg in ([(par, negrita)] if isinstance(par, str) else par):
            r = p.add_run()
            r.text = trozo
            r.font.size, r.font.bold = Pt(tam), neg
            r.font.color.rgb = color
    return caja


def diapositiva(titulo: str, subtitulo: str = None, seccion: str = None):
    """Lamina con seccion (arriba), titular (el hallazgo) y bajada."""
    d = prs.slides.add_slide(prs.slide_layouts[6])
    if seccion:
        caja_texto(d, MARGEN, Inches(0.12), UTIL_W, Inches(0.25), seccion.upper(), tam=9, color=ACENTO, negrita=True)
    caja_texto(d, MARGEN, Inches(0.32), UTIL_W, Inches(0.8), titulo, tam=19, negrita=True)
    if subtitulo:
        caja_texto(d, MARGEN, Inches(1.13), UTIL_W, Inches(0.4), subtitulo, tam=12, color=TINTA_2)
    return d


def portada(titulo: str, linea_1: str, linea_2: str, color: str):
    """Titulo, barra de color, dos lineas de contexto y el area al pie."""
    d = prs.slides.add_slide(prs.slide_layouts[6])
    caja_texto(d, MARGEN, Inches(2.2), UTIL_W, Inches(0.9), titulo, tam=28, negrita=True)
    barra = d.shapes.add_shape(1, MARGEN, Inches(3.2), Inches(1.2), Inches(0.06))
    barra.fill.solid()
    barra.fill.fore_color.rgb = rgb(color)
    barra.line.fill.background()
    caja_texto(d, MARGEN, Inches(3.4), UTIL_W, Inches(0.4), linea_1, tam=18, color=TINTA_2)
    caja_texto(d, MARGEN, Inches(4.0), UTIL_W, Inches(0.34), linea_2, tam=14, color=TINTA_2)
    caja_texto(d, MARGEN, Inches(6.5), UTIL_W, Inches(0.4), "Sistemas de Información", tam=12, color=TINTA_3)


def nombre_sociedad(soc: str) -> str:
    """Nombre de la region para las portadas."""
    return soc


def imagen(d, fig, x=MARGEN, y=CUERPO_Y, w=UTIL_W, h=CUERPO_H):
    """Figura ajustada a la caja w x h, sin deformarla y centrada en horizontal."""
    buf = io.BytesIO()
    fig.savefig(buf, format="png", dpi=170, bbox_inches="tight", facecolor="white")
    plt.close(fig)
    buf.seek(0)
    pic = d.shapes.add_picture(buf, x, y)
    escala = min(w / pic.width, h / pic.height)
    pic.width, pic.height = int(pic.width * escala), int(pic.height * escala)
    pic.left = int(x + (w - pic.width) / 2)
    return pic


def color_calor(k: float) -> RGBColor:
    """Escala de un solo tono: blanco (k=0) a naranja (k=1), con texto oscuro legible en todo el rango."""
    a, b = (255, 255, 255), (0xE0, 0x6A, 0x2E)
    k = 0.06 + 0.7 * max(0.0, min(1.0, k))
    return RGBColor(*[int(x + (y - x) * k) for x, y in zip(a, b)])


def tabla(d, df: pd.DataFrame, x=MARGEN, y=CUERPO_Y, w=UTIL_W, tam=10, ancho_indice=0.25,
          negrita_filas=(), alto_fila=None, calor=None, anchos=None):
    """Tabla nativa con el texto ya formateado. Columnas de 2 niveles -> 2 filas de encabezado (combinadas).
    calor: DataFrame numerico del mismo tamaño; las columnas con valores se pintan como mapa de calor. La escala es
    por columna, o por indicador (todos sus años juntos) si las columnas tienen 2 niveles.
    anchos: ancho de cada columna en pulgadas, incluida la del indice (si no, se reparten por igual)."""
    if calor is not None:
        calor = calor.astype(float).copy()
        grupos = (calor.columns.get_level_values(0) if isinstance(calor.columns, pd.MultiIndex)
                  else calor.columns)
        for g in pd.unique(grupos):
            cols_g = [col for col, gg in zip(calor.columns, grupos) if gg == g]
            bloque = calor[cols_g]
            lo, hi = bloque.min().min(), bloque.max().max()
            calor[cols_g] = (bloque - lo) / ((hi - lo) or 1)
    dos = isinstance(df.columns, pd.MultiIndex)
    n_enc = 2 if dos else 1
    n_f, n_c = df.shape[0] + n_enc, df.shape[1] + 1
    alto_fila = alto_fila or Inches(0.08 + tam * 0.021)
    forma = d.shapes.add_table(n_f, n_c, x, y, w, alto_fila * n_f)
    t = forma.table
    t.first_row, t.horz_banding = True, False
    t.columns[0].width = int(w * ancho_indice)
    for j in range(1, n_c):
        t.columns[j].width = int(w * (1 - ancho_indice) / (n_c - 1))
    for j, a_ in enumerate(anchos or []):
        t.columns[j].width = Inches(a_)
    for i in range(n_f):
        t.rows[i].height = alto_fila
        for j in range(n_c):
            celda_ = t.cell(i, j)
            if i < n_enc:
                if j == 0:
                    texto_ = (df.index.name or "") if i == n_enc - 1 else ""
                elif dos:
                    texto_ = str(df.columns[j - 1][i])
                else:
                    texto_ = str(df.columns[j - 1])
            else:
                texto_ = str(df.index[i - n_enc]) if j == 0 else str(df.iat[i - n_enc, j - 1])
            celda_.text = texto_
            celda_.fill.solid()
            celda_.fill.fore_color.rgb = FONDO_ENC if i < n_enc else BLANCO
            if calor is not None and i >= n_enc and j > 0 and pd.notna(calor.iat[i - n_enc, j - 1]):
                celda_.fill.fore_color.rgb = color_calor(calor.iat[i - n_enc, j - 1])
            celda_.vertical_anchor = MSO_ANCHOR.MIDDLE
            celda_.margin_left = celda_.margin_right = Inches(0.04)
            celda_.margin_top = celda_.margin_bottom = Inches(0.01)
            p = celda_.text_frame.paragraphs[0]
            p.alignment = PP_ALIGN.LEFT if j == 0 else (PP_ALIGN.CENTER if i < n_enc else PP_ALIGN.RIGHT)
            for r in p.runs:
                r.font.size = Pt(tam)
                r.font.bold = i < n_enc or df.index[i - n_enc] in negrita_filas
                r.font.color.rgb = TINTA
    if dos:  # combina el encabezado superior repetido
        j = 1
        while j < n_c:
            k = j
            while k + 1 < n_c and df.columns[k][0] == df.columns[j - 1][0]:
                k += 1
            if k > j:
                for jj in range(j + 1, k + 1):   # vaciar antes de combinar: merge junta los textos
                    t.cell(0, jj).text = ""
                t.cell(0, j).merge(t.cell(0, k))
            j = k + 1
    return forma


def a_texto(t: pd.DataFrame, filas_indicador=False) -> pd.DataFrame:
    """Formatea una tabla: por fila (indicadores en filas) o por columna (nivel 0)."""
    out = t.astype(object).copy()
    for c in t.columns:
        for i in t.index:
            ind = i if filas_indicador else (c[0] if isinstance(c, tuple) else c)
            v = t.at[i, c]
            if isinstance(v, str):          # columnas de texto (p. ej. canal de venta)
                continue
            out.at[i, c] = variacion(v, ind.startswith("%")) if es_variacion(c) else formato_de(ind)(v)
    return out


def corto(ind: str) -> str:
    """Rotulo corto para encabezados de tabla."""
    return ind.replace("% aplicado al precio de lista", "% aplicado").replace("de precio de mercado", "precio mercado")

## C.2 Informe general por sociedad

Un PowerPoint por sociedad, con la estructura del informe revisado (`old/… (v2).pptx`): portada y las secciones 1 a 9
de la parte B, una lámina tras otra (sin separadores), con número de lámina y sin pie de fuente.

| Sección | Láminas |
|---|---|
| 1. Contexto del alcance | 1 |
| 2. Panorama 2025–2026 | tabla + 2 de evolución mes a mes |
| 3. Rangos de % aplicado al precio de lista | gráfica + tabla del total |
| 4–6. Rangos por canal de distribución, canal de venta y grupo de material 6 | resumen + gráfica y tabla de cada uno (`TOP_PPT` grupos de material) |
| 7. Canal de venta y canal de distribución | una por canal de venta |
| 8. Concentración del ajuste – Distribuidoras | 1 |
| 9. Concentración del ajuste – Clientes (mes de `MES_CLIENTES`) | Mayoreo y Puntos de Venta |

In [ ]:
SECCIONES = {
    1: "Contexto del alcance",
    2: f"Panorama {A1}–{A2}",
    3: "Rangos de % aplicado al precio de lista",
    4: "Rangos de % de lista precio · Canal de distribución",
    5: "Rangos de % de lista precio · Canal de venta",
    6: "Rangos de % de lista precio · Grupo material 6",
    7: "Canal de venta y canal de distribución",
    8: "Concentración del ajuste - Distribuidoras",
}
SECCION_DIM = {"Canal de distribución": 4, "Canal de venta": 5, "Grupo de material 6": 6}
ALCANCE = ("Inicialmente el foco fue el ajuste de fecha corta, pero fruto del primer modelo análisis de datos, se "
           "incorporó al alcance el porcentaje de cumplimiento de listas, con el ajuste general: en torno al ajuste "
           "precio de mercado y fecha corta.")
COLS_RES = [VENTA_KG, VENTA_BRUTA, P_GEN, *PILAR]
COLS_CONC = [P_GEN, "% acumulado ajuste general", VENTA_KG, VENTA_BRUTA, L_GEN, D_GEN, L_AJ, D_AJ, L_FC, D_FC]
ANIOS_CONC = {L_GEN: (A1, A2), D_FC: (A1, A2), L_FC: (A1, A2)}
ANCHOS_CONC = [1.51, 1.15, 0.65, 0.76, 1.01] + [0.81] * 9   # pulgadas, como en el informe revisado


def sec(n: int) -> str:
    return f"{n}. {SECCIONES[n]}"


def sec_clientes(canal: str) -> str:
    return (f"9. Concentración del ajuste – Clientes {ROTULO_CANAL.get(canal, canal)} – "
            f"Análisis de {MES_TXT}")


def lamina(titulo: str = None, subtitulo: str = None, seccion: str = None, compacta=False):
    """Lamina del informe general: seccion (azul, 16), titular (de 19 a 14 segun el largo, para que quepa en una
    linea), bajada y numero de lamina. compacta: todo mas arriba, para tablas largas."""
    d = prs.slides.add_slide(prs.slide_layouts[6])
    y_sec, y_tit, y_sub = (0.08, 0.41, 0.75) if compacta else (0.25, 0.62, 1.13)
    caja_texto(d, MARGEN, Inches(y_sec), UTIL_W, Inches(0.37), seccion.upper(), tam=16, color=ACENTO, negrita=True)
    if titulo:
        tam = next((t_ for t_ in (19, 16, 14) if len(titulo) <= 1770 / t_), 14)
        caja_texto(d, MARGEN, Inches(y_tit), UTIL_W, Inches(0.4), titulo, tam=16 if compacta else tam, negrita=True)
    if subtitulo:
        caja_texto(d, MARGEN, Inches(y_sub), UTIL_W, Inches(0.4), subtitulo, tam=12, color=TINTA_2)
    caja_texto(d, ANCHO - MARGEN - Inches(0.6), Inches(7.1), Inches(0.6), Inches(0.24), str(len(prs.slides)),
               tam=8, color=TINTA_3, alinear=PP_ALIGN.RIGHT)
    return d


def lamina_rangos(dim, vista, soc, m, seccion):
    """Dos laminas: la grafica (con % y ajuste al lado de cada barra) y la tabla del detalle por rango."""
    c = cubo_de(dim, soc, m)
    nombre = vista if dim == "Total" else f"{m} ({vista})"
    ind = indicadores(c.groupby(level="Año").sum()).reindex(ANIOS)  # un miembro puede no tener venta un año
    bajada = (f"% aplicado al precio de lista general {pct(ind.loc[A1, L_GEN])} → {pct(ind.loc[A2, L_GEN])} · "
              f"ajuste de precio de mercado {pct(ind.loc[A1, L_AJ])} → {pct(ind.loc[A2, L_AJ])} · "
              f"fecha corta {pct(ind.loc[A1, L_FC])} → {pct(ind.loc[A2, L_FC])}")
    d = lamina(f"{nombre}: {frase_rango(c)}", bajada, seccion)
    imagen(d, figura_rangos(c, f"% de la venta bruta por rango de % aplicado al precio de lista general y motivo — "
                               f"{nombre} (al lado de cada barra: % y ajuste general)", figsize=(14, 7.4)))
    tr = tabla_rangos(c)
    txt = a_texto(tr)
    txt.columns = pd.MultiIndex.from_tuples([(corto(i), a_) for i, a_ in tr.columns])
    txt.index.name = "Rango"
    d = lamina(f"{nombre}: participación y ajuste en pesos en cada rango",
               f"{PERIODO} {A1} vs {A2} · rango de % aplicado al precio de lista general", seccion)
    tabla(d, txt, y=Inches(1.55), tam=9, ancho_indice=0.16, alto_fila=Inches(0.205))


def lamina_resumen_dim(dim, vista, soc, seccion):
    """Dos tablas: arriba volumen y pesos, abajo los tres % aplicados con sus ajustes."""
    res = resumen_dim(dim, soc, COLS_RES)
    mayor = res[(L_GEN, str(A2))].sub(res[(L_GEN, str(A1))]).idxmin()
    compacta = len(res) > 5                       # muchas filas: todo mas arriba y letra mas pequeña
    d = lamina(f"{dim}: {mayor} tuvo la mayor caída del % aplicado al precio de lista general "
               f"({pct(res.loc[mayor, (L_GEN, str(A1))])} → {pct(res.loc[mayor, (L_GEN, str(A2))])})",
               f"{vista} · venta kg, venta bruta y los tres % aplicados al precio de lista con sus ajustes — "
               f"{PERIODO} {A1} vs {A2}", seccion, compacta=compacta)
    alto, y_ = Inches(0.3), Inches(1.2 if compacta else 1.6)
    for cols_, ancho, a_idx, tam in (([VENTA_KG, VENTA_BRUTA, P_GEN], Inches(10.0), 0.22, 10 if compacta else 10.5),
                                     (PILAR, UTIL_W, 0.16, 10 if compacta else 11)):
        sub = res[[col for col in res.columns if col[0] in cols_]]
        txt = a_texto(sub)
        txt.columns = pd.MultiIndex.from_tuples([(corto(i), a_) for i, a_ in sub.columns])
        txt.index = [str(i)[:32] for i in txt.index]
        txt.index.name = dim
        calor = None
        if dim in CON_CALOR:
            calor = sub[[col for col in sub.columns if col[0].startswith("% Ajuste")]].reindex(columns=sub.columns)
        tabla(d, txt, y=y_, w=ancho, tam=tam, ancho_indice=a_idx, alto_fila=alto, calor=calor)
        y_ += alto * (len(txt) + 2) + Inches(0.2)


def tabla_conc(t: pd.DataFrame, nombre: str) -> pd.DataFrame:
    cols = [(ind, str(a_)) for ind in COLS_CONC for a_ in ANIOS_CONC.get(ind, (A2,))]
    out = a_texto(t[cols])
    out.columns = pd.MultiIndex.from_tuples([(corto(i).replace("% acumulado ajuste general", "% acum."), a_)
                                             for i, a_ in cols])
    out.index = [str(i)[:40] for i in out.index]
    out.index.name = nombre
    return out


def construir_informe(vista: str):
    soc = VISTAS[vista]
    nueva_presentacion()

    # ---------------------------------------------------------------------- portada
    portada("Diagnóstico de Ajustes y % Cumplimiento Aplicado a Lista de Precios",
            f"{A1} vs {A2} · acumulado {PERIODO} · {nombre_sociedad(soc)}",
            f"Todos los canales · datos a {FECHA_MAX:%d/%m/%Y}", COLOR_SOC[soc])

    # ---------------------------------------------------------------------- 1. contexto del alcance
    d = prs.slides.add_slide(prs.slide_layouts[6])
    caja_texto(d, MARGEN, Inches(0.46), UTIL_W, Inches(0.37), sec(1).upper(), tam=16, color=ACENTO, negrita=True)
    caja_texto(d, MARGEN, Inches(1.0), UTIL_W, Inches(0.57), ALCANCE, tam=14)
    pilar = pd.DataFrame([
        ["neta / bruta", "ajuste + fecha corta"],
        ["(bruta − ajuste) / bruta", "solo el ajuste de precio de mercado"],
        ["(bruta − fecha corta) / bruta", "solo la fecha corta (rotación)"],
    ], index=["% Aplicado al precio de lista general  |  % Ajuste general",
              "% Aplicado al precio de lista por ajuste de precio de mercado  |  % Ajuste de precio de mercado",
              "% Aplicado al precio de lista por fecha corta  |  % Ajuste de fecha corta"],
       columns=["% de lista precio", "% de ajuste"])
    pilar.index.name = "El pilar: % Aplicado al precio de lista + % de ajuste = 100%"
    tabla(d, pilar, y=Inches(2.05), tam=12, ancho_indice=0.46, alto_fila=Inches(0.42))
    caja_texto(d, MARGEN, Inches(4.31), UTIL_W, Inches(1.9), [
        [("Venta bruta: ", True), ("lo que se habría cobrado a precio de lista (venta neta + ajustes). Es la base de "
                                   "todas las participaciones.", False)],
        [("Venta neta: ", True), ("Valor facturado;", False)],
        [("Venta kg: ", True), ("kilos vendidos.", False)],
        [("Rangos: ", True), ("22 rangos del % Aplicado al precio de lista Cada venta (día × cliente × grupo de "
                              "material × distribuidora) cae en uno.", False)],
        [("Tipos de Ajuste: ", True), ("Ajuste de fecha corta (rotación) y ajuste de precio de mercado", False)],
        [("Periodos: ", True), (f"acumulado {PERIODO} {A1} vs {A2}", False)],
    ], tam=13, espacio=7)

    # ---------------------------------------------------------------------- 2. panorama
    pan = a_texto(PANORAMA[vista], filas_indicador=True)
    pan.index.name = "Indicador"
    d = lamina(f"La lista por kilo {sube_baja(ch_(vista, PRECIO_LISTA))} {num(abs(ch_(vista, PRECIO_LISTA)))}%, "
               f"y el neto por kilo {sube_baja(ch_(vista, PRECIO_NETO))} {num(abs(ch_(vista, PRECIO_NETO)))}%",
               f"% aplicado al precio de lista general {de_a(vista, L_GEN)} · ajuste de precio de mercado {de_a(vista, L_AJ)} · "
               f"fecha corta {de_a(vista, L_FC)}", sec(2))
    tabla(d, pan, x=Inches(1.5), w=Inches(10.3), tam=14, ancho_indice=0.46,
          negrita_filas=[PRECIO_LISTA, PRECIO_NETO, L_GEN, L_AJ, L_FC], alto_fila=Inches(0.33))
    d = lamina(seccion=sec(2))
    imagen(d, figura_evolucion(soc, ("kg", "pesos"), figsize=(15, 7.2)), y=Inches(1.0), h=Inches(5.8))
    pm = indicadores(mensual_de(soc))
    d = lamina(f"El % aplicado al precio de lista general pasó de {pct(pm[L_GEN].iloc[0])} (enero {A1}) a "
               f"{pct(pm[L_GEN].iloc[-1])} ({MES_FIN_TXT} {A2})", seccion=sec(2))
    imagen(d, figura_evolucion(soc, ("precio", "aplicado"), figsize=(15, 7.2)), y=Inches(1.15), h=Inches(5.75))

    # ---------------------------------------------------------------------- 3. rangos del total
    lamina_rangos("Total", vista, soc, "Total", sec(3))

    # ---------------------------------------------------------------------- 4 a 6. rangos por dimension
    for dim, n in SECCION_DIM.items():
        lamina_resumen_dim(dim, vista, soc, sec(n))
        tope = TOP_PPT if dim == "Grupo de material 6" else None
        for m in miembros(dim, soc)[:tope]:
            lamina_rangos(dim, vista, soc, m, sec(n))

    # ---------------------------------------------------------------------- 7. canal de venta x canal de distribucion
    for dim in CRUCES:
        for m in miembros(dim, soc):
            t = desglose_canal(dim, soc, m)
            canales = [x for x in t.columns.unique(1) if x != "Total"]
            kg2 = t.loc[VENTA_KG, str(A2)]
            if len(canales) > 1:
                mayor = max(canales, key=lambda x: kg2.get(x, 0) if pd.notna(kg2.get(x)) else 0)
                otro = next(x for x in canales if x != mayor)
                titulo = (f"{m}: {mayor} es el {pct(kg2[mayor] / kg2['Total'] * 100)} de la venta kg; "
                          f"% aplicado al precio de lista general {pct(t.loc[L_GEN, (str(A2), mayor)])} en {mayor} "
                          f"y {pct(t.loc[L_GEN, (str(A2), otro)])} en {otro} ({A2})")
            else:
                titulo = f"{m}: se vende solo por {canales[0]}"
            txt = a_texto(t, filas_indicador=True)
            txt.index.name = "Indicador"
            d = lamina(titulo, f"{vista} · {dim} · Mayoreo + Puntos de Venta = Total (kg, venta y ajustes); "
                               f"los % y precios se recalculan · {PERIODO} {A1} vs {A2}", sec(7))
            tabla(d, txt, tam=10, ancho_indice=0.27, alto_fila=Inches(0.33), negrita_filas=[L_GEN, L_AJ, L_FC])

    # ---------------------------------------------------------------------- 8. distribuidoras
    mitad = mitad_distribuidoras(soc)
    d = lamina(f"{len(mitad)} distribuidoras concentran la mitad del ajuste general — "
               + ", ".join(n_.title() for n_ in mitad[:6]),
               f"{vista} · ordenadas por ajuste general {PERIODO} {A2}", sec(8))
    tabla(d, tabla_conc(distribuidoras(soc).head(FILAS_PPT), "Distribuidora"), tam=9,
          negrita_filas=[str(i)[:40] for i in mitad], alto_fila=Inches(0.33), anchos=ANCHOS_CONC)

    # ---------------------------------------------------------------------- 9. clientes del mes
    for canal in CANALES:
        tc = top_clientes(soc, canal)
        if tc.empty:
            continue
        cc = concentracion(soc, canal)
        n_ = cc.loc["Clientes con el 50% del ajuste general", str(A2)]
        txt = a_texto(tc)
        txt.columns = [corto(col) for col in tc.columns]
        txt.index = [str(i)[:38] for i in txt.index]
        txt.index.name = "Cliente"
        d = lamina(f"Clientes {canal}: {num(n_, 0)} concentran la mitad del ajuste general de {MES_TXT} {A2}",
                   f"{vista} · top {TOP_CLIENTES} por ajuste general en {MES_TXT} {A2} · "
                   f"{num(cc.loc['Clientes con ajuste', str(A2)], 0)} clientes con ajuste", sec_clientes(canal))
        calor = tc[[col for col in tc.columns if col.startswith("% Ajuste")]].reindex(columns=tc.columns)
        calor.index = txt.index
        tabla(d, txt, y=Inches(1.5), tam=7.5, ancho_indice=0.22, alto_fila=Inches(0.166), calor=calor)

    guardar(f"Informe_Ajustes_{ARCHIVO[vista]}_{A1}_vs_{A2}.pptx")

### Región Oriente

In [ ]:
construir_informe("Región Oriente")

### Región Occidente

In [ ]:
construir_informe("Región Occidente")

## C.3 Top de clientes por ajuste general

Mes de `MES_CLIENTES`. Cuatro PowerPoint, con portada y sin separadores: por sociedad, uno **por distribuidora** (de
mayor a menor ajuste general del mes) y uno **por canal de venta** (tradicional, institucional, modernos y amas de
casa), cada grupo abierto en Mayoreo y Puntos de Venta. Cada lámina trae los `TOP_CLIENTES` con más ajuste general en
pesos y el titular dice qué parte del ajuste del grupo suman. Se omiten los grupos sin clientes con ajuste.

In [ ]:
CANALES_DIST_TOP = {"Mayoreo": "Mayoreo", "Puntos Venta": "Puntos de Venta"}   # valor en la base -> rotulo


def filtrar_top(soc, filtros=None) -> pd.DataFrame:
    k = clientes_det[clientes_det.index.get_level_values("Sociedad") == soc]
    for nivel, valor in (filtros or {}).items():
        k = k[k.index.get_level_values(nivel) == valor]
    return k


def distribuidoras_top(soc) -> list:
    """Distribuidoras de la sociedad, de mayor a menor ajuste general del mes."""
    k = filtrar_top(soc)
    g = (k["Descuento Ajuste Precio Mercado"] + k["Descuento Rotacion / Fecha Corta"]).groupby(
        level="Nombre Distribuidora").sum()
    return list(g[g > 0].sort_values(ascending=False).index)


def grupos_top(soc, tipo: str):
    """(rotulo, filtros) de cada grupo: por canal de venta o por distribuidora, abierto en Mayoreo y Puntos de Venta."""
    principales = CANALES_VENTA_TOP if tipo == "canal" else distribuidoras_top(soc)
    nivel = "Canal de Venta" if tipo == "canal" else "Nombre Distribuidora"
    for p in principales:
        for canal, rotulo in CANALES_DIST_TOP.items():
            yield f"{p if tipo == 'canal' else p.title()} · {rotulo}", {nivel: p, "Canal de Distribucion": canal}


def lamina_top(t, part, total, n_cli, vista, grupo, sin_columna):
    t = t[[c_ for c_ in COLS_TOP if c_ != sin_columna]]
    txt = a_texto(t)
    txt.columns = [corto(c_) for c_ in t.columns]
    txt.index = [str(i)[:36] for i in txt.index]
    txt.index.name = "Cliente"
    calor = t[[D_GEN, D_AJ, D_FC]].reindex(columns=t.columns)
    calor.index = txt.index
    sujeto = "el único cliente con ajuste concentra" if len(t) == 1 else f"los {len(t)} clientes con más ajuste general suman"
    d = diapositiva(f"{grupo}: {sujeto} el {pct(part)} de ese ajuste",
                    f"{vista} · {MES_TXT} {A2} · ajuste general del grupo: {pesos(total)} · {num(n_cli, 0)} clientes "
                    "con ajuste · ordenados por ajuste general en pesos")
    tabla(d, txt, y=Inches(1.5), tam=7, ancho_indice=0.22, alto_fila=Inches(0.17), calor=calor)


def informe_top(vista: str, tipo: str):
    """Un PowerPoint: top de cada grupo (canal de venta o distribuidora) x Mayoreo / Puntos de Venta."""
    soc = VISTAS[vista]
    nueva_presentacion()
    portada("Top de clientes por ajuste general",
            f"{MES_TXT.capitalize()} {A2} · por {'distribuidora' if tipo == 'distribuidora' else 'canal de venta'} · "
            f"{nombre_sociedad(soc)}",
            f"Mayoreo y Puntos de Venta · top {TOP_CLIENTES} por grupo, ordenados por ajuste general en pesos",
            COLOR_SOC[soc])
    for grupo, filtros in grupos_top(soc, tipo):
        t, part, total, n_cli = top_de(filtrar_top(soc, filtros))
        if t is not None:
            # por distribuidora ya se sabe la distribuidora; por canal de venta ya se sabe el canal
            lamina_top(t, part, total, n_cli, vista, grupo,
                       sin_columna="Distribuidora" if tipo == "distribuidora" else "Canal de venta")
    nombre = "Distribuidoras" if tipo == "distribuidora" else "Canales_de_venta"
    guardar(f"Top_Clientes_{ARCHIVO[vista]}_{nombre}_{MES_TXT.capitalize()}_{A2}.pptx")


for vista in VISTAS:
    for tipo in ("distribuidora", "canal"):
        informe_top(vista, tipo)

## C.4 Informe de un cliente

Cuatro láminas para `CLIENTE` (A.1), desde sus líneas de factura (enero 2025 al último mes con datos):

1. **Resumen**: la tabla comparativa del acumulado 2025 vs 2026 (igual a la de las sociedades).
2. **Tendencia del % de ajuste** mes a mes (fecha corta + ajuste de precio de mercado = general) y la venta bruta vs neta.
3. **Materiales con más ajuste**: los `TOP_MATERIALES` de mayor ajuste general en el último año, 2025 vs 2026.
4. **Mes a mes**: la tabla mensual con mapa de calor en los % de ajuste.

In [ ]:
SQL_CLIENTE = """
SELECT v.date AS fecha, v.bill AS factura, v.material, m.name AS producto, g6.name AS grupo_material,
       cdd.company_code AS sociedad, regexp_replace(cdd.new_zone_name, '^[^-]*-', '') AS distribuidora,
       dc.distribution_channel_name AS canal, v.kilogram AS kg, v.value AS neta,
       v.value + COALESCE(v.discount_value, 0) AS bruta,
       COALESCE(v.discount_market_price_adjustment, 0) AS aj,
       COALESCE(v.short_dated_discount, 0) AS fc
FROM public.daily_sales_sector_10_15_20_21 v
LEFT JOIN public.materials m ON v.material::integer = m.code
LEFT JOIN public.material_groups_6 g6 ON g6.code = m.gr6
LEFT JOIN public.commercial_distributors_details cdd ON v.id_commercial_distributors = cdd.id
LEFT JOIN public.distribution_channels dc ON v.distribution_channel = dc.distribution_channel
WHERE v.client = :cliente AND v.date >= :ini AND v.date < :fin
"""
with ENGINE.connect() as c:
    cl = pd.read_sql(text(SQL_CLIENTE), c, params={"cliente": CLIENTE, "ini": f"{A1}-01-01",
                                                   "fin": (FECHA_MAX + pd.Timedelta(days=1)).strftime("%Y-%m-%d")})
for col in ["kg", "neta", "bruta", "aj", "fc"]:
    cl[col] = cl[col].astype(float)
cl["fecha"] = pd.to_datetime(cl["fecha"])
cl["gen"] = cl["aj"] + cl["fc"]
cl["Mes"] = cl["fecha"].dt.to_period("M")
cl["Año"] = cl["fecha"].dt.year
NOMBRE_CLI = nombre_cliente.get(CLIENTE, CLIENTE).strip()
SOC_CLI = SOCIEDADES.get(str(cl["sociedad"].mode().iloc[0]), "")


def ind_cliente(df, por) -> pd.DataFrame:
    """Indicadores del cliente desde sumas."""
    g = df.groupby(por)[["kg", "bruta", "neta", "aj", "fc", "gen"]].sum()
    b = g["bruta"].where(g["bruta"] != 0)
    kg = g["kg"].where(g["kg"] > 0)
    return pd.DataFrame({
        "Facturas": df.groupby(por)["factura"].nunique(),
        VENTA_KG: g["kg"],
        VENTA_BRUTA: g["bruta"], VENTA_NETA: g["neta"], P_GEN: g["gen"],
        P_AJ: g["aj"], P_FC: g["fc"],
        D_GEN: g["gen"] / b * 100, D_AJ: g["aj"] / b * 100, D_FC: g["fc"] / b * 100,
        PRECIO_LISTA: g["bruta"] / kg, PRECIO_NETO: g["neta"] / kg,
    })


acum_cli = cl[cl["fecha"].dt.month.isin(MESES_ACUM)]
anual_cli = ind_cliente(acum_cli, "Año").reindex(ANIOS)
mes_cli_t = ind_cliente(cl, "Mes")
kg_fc = acum_cli[acum_cli["fc"] > 0].groupby("Año")["kg"].sum() / acum_cli.groupby("Año")["kg"].sum() * 100
# materiales: top por ajuste general del ultimo año
mat_cli = ind_cliente(acum_cli, ["producto", "Año"])
top_mat = (mat_cli.xs(A2, level="Año")[P_GEN].sort_values(ascending=False).head(TOP_MATERIALES).index
           if A2 in mat_cli.index.get_level_values("Año") else [])
print(f"{CLIENTE} - {NOMBRE_CLI} · {SOC_CLI} · {len(cl):,} líneas · {cl['factura'].nunique():,} facturas")

In [ ]:
def figura_tendencia_cliente(figsize=(14, 7.6)):
    """Barras apiladas del % de ajuste por mes (fecha corta + ajuste = general) y venta bruta vs neta (la franja es
    el ajuste)."""
    t = mes_cli_t
    x = list(range(len(t)))
    etiquetas = t.index.strftime("%Y-%m")
    corte = sum(t.index.year == A1) - 0.5
    fig, (a1, a2) = plt.subplots(2, 1, figsize=figsize, sharex=True, gridspec_kw={"height_ratios": [1.7, 1]})
    a1.bar(x, t[D_FC], color=MOTIVOS["Con fecha corta"], width=0.75, edgecolor="white", linewidth=1, label="Fecha corta")
    a1.bar(x, t[D_AJ], bottom=t[D_FC], color=MOTIVOS["Solo ajuste de precio de mercado"], width=0.75, edgecolor="white", linewidth=1,
           label="Ajuste de precio de mercado")
    for i, v in enumerate(t[D_GEN]):
        a1.annotate(pct(v), (i, v), xytext=(0, 3), textcoords="offset points", ha="center", fontsize=8,
                    color="#0b0b0b", fontweight="bold")
    a1.set_ylim(0, max(100, t[D_GEN].max() * 1.12))
    a1.set_title("% de ajuste sobre la venta bruta, por tipo (la suma es el % de ajuste general)", loc="left", fontsize=11)
    a1.legend(frameon=False, fontsize=9, loc="upper left", bbox_to_anchor=(1, 1))
    color_ = COLOR_SOC.get(SOC_CLI, "#2a78d6")
    a2.fill_between(x, t[VENTA_NETA], t[VENTA_BRUTA], color=TINTE_SOC.get(SOC_CLI, "#a9c9f0"), alpha=0.6,
                    linewidth=0, label="Ajuste")
    a2.plot(x, t[VENTA_BRUTA], color=color_, linewidth=2, marker="o", markersize=4, label="Venta bruta")
    a2.plot(x, t[VENTA_NETA], color=color_, linewidth=2, linestyle="--", marker="o", markersize=4, label="Venta neta")
    for i in (0, len(x) - 1):
        a2.annotate(pesos(t[VENTA_BRUTA].iloc[i]), (x[i], t[VENTA_BRUTA].iloc[i]), xytext=(0, 6),
                    textcoords="offset points", ha="center", fontsize=8, color="#0b0b0b")
        a2.annotate(pesos(t[VENTA_NETA].iloc[i]), (x[i], t[VENTA_NETA].iloc[i]), xytext=(0, -10),
                    textcoords="offset points", ha="center", va="top", fontsize=8, color="#0b0b0b")
    a2.margins(y=0.25)
    a2.legend(frameon=False, fontsize=9, loc="upper left", bbox_to_anchor=(1, 1))
    a2.set_title("Venta bruta y venta neta por mes ($) — la franja es el ajuste", loc="left", fontsize=11)
    for ax, f in ((a1, lambda v, _: f"{v:,.0f}%"), (a2, lambda v, _: pesos(v))):
        ax.axvline(corte, color="#8c8b87", linestyle=":", linewidth=1)
        ejes_limpios(ax, formato_y=f)
    a2.set_xticks(x, etiquetas, rotation=90)
    plt.tight_layout()
    return fig


def figura_materiales_cliente(figsize=(14, 7)):
    """Top materiales por ajuste general: % de ajuste por tipo (barras apiladas), un panel por año; al lado % y $."""
    fig, ejes = plt.subplots(1, len(ANIOS), figsize=figsize, sharey=True, sharex=True)
    nombres = [str(m).title()[:34] for m in top_mat]
    for ax, anio in zip(ejes, ANIOS):
        t = mat_cli[mat_cli.index.get_level_values("Año") == anio].droplevel("Año").reindex(top_mat)
        fc, aj = t[D_FC].fillna(0), t[D_AJ].fillna(0)
        ax.barh(nombres, fc, color=MOTIVOS["Con fecha corta"], height=0.7, edgecolor="white", label="Fecha corta")
        ax.barh(nombres, aj, left=fc, color=MOTIVOS["Solo ajuste de precio de mercado"], height=0.7, edgecolor="white", label="Ajuste de precio de mercado")
        for i, (g_, plata, kg_) in enumerate(zip(t[D_GEN], t[P_GEN], t[VENTA_KG])):
            texto_ = "sin compras" if pd.isna(g_) else f"{pct(g_)} · {pesos(plata)} · {num(kg_, 0)} kg"
            ax.annotate(texto_, (0 if pd.isna(g_) else g_, i), xytext=(4, 0), textcoords="offset points",
                        va="center", fontsize=7.5, color="#52514e")
        ax.set_xlim(0, 175)
        ax.set_title(str(anio), loc="left", fontsize=12)
        ejes_limpios(ax, formato_x=lambda v, _: f"{v:,.0f}%" if v <= 100 else "")
    ejes[0].invert_yaxis()
    ejes[-1].legend(frameon=False, fontsize=9, loc="lower right")
    fig.suptitle(f"% de ajuste por tipo · al lado: % ajuste general · ajuste en pesos · venta kg ({PERIODO})",
                 x=0.01, ha="left", fontsize=11)
    plt.tight_layout()
    return fig


def tabla_mensual_cliente() -> pd.DataFrame:
    """Mes a mes + totales del acumulado de cada año."""
    t = mes_cli_t.copy()
    t.index = t.index.strftime("%Y-%m")
    tot = anual_cli.copy()
    tot.index = [f"{PERIODO.capitalize()} {a_}" for a_ in tot.index]
    return pd.concat([t, tot])[["Facturas", VENTA_KG, VENTA_BRUTA, VENTA_NETA, P_GEN,
                                D_GEN, D_AJ, D_FC, PRECIO_LISTA, PRECIO_NETO]]


def construir_informe_cliente():
    nueva_presentacion()
    a1, a2 = anual_cli.loc[A1], anual_cli.loc[A2]
    seccion = f"{CLIENTE} · {NOMBRE_CLI}"

    # ------------------------------------------------------------------ 1. resumen
    d = diapositiva(f"{NOMBRE_CLI.title()}: el {pct(kg_fc.get(A2, 0))} de lo que compra es fecha corta y paga "
                    f"{pct(100 - a2[D_GEN])} de la lista",
                    f"{SOC_CLI} · {', '.join(cl['distribuidora'].dropna().unique()[:2])} · "
                    f"{', '.join(cl['canal'].dropna().unique()[:2])} · {PERIODO} {A2} vs {A1}", seccion)
    # misma tabla comparativa del panorama de las sociedades
    sumas = (acum_cli.rename(columns={"kg": "Venta Kg", "bruta": "Venta Bruta", "neta": "Venta Neta",
                                      "aj": "Descuento Ajuste Precio Mercado", "fc": "Descuento Rotacion / Fecha Corta"})
             .groupby("Año")[MEDIDAS].sum().reindex(ANIOS))
    pan = a_texto(comparar(indicadores(sumas)), filas_indicador=True)
    pan.index.name = "Indicador"
    tabla(d, pan, x=Inches(1.5), w=Inches(10.3), tam=11, ancho_indice=0.42,
          negrita_filas=[PRECIO_LISTA, PRECIO_NETO, L_GEN, L_AJ, L_FC], alto_fila=Inches(0.33))

    # ------------------------------------------------------------------ 2. tendencia
    m2 = mes_cli_t[mes_cli_t.index.year == A2]
    d = diapositiva(f"El % de ajuste pasó de {pct(a1[D_GEN])} a {pct(a2[D_GEN])}; el máximo fue "
                    f"{pct(m2[D_GEN].max())} en {MESES_ES[m2[D_GEN].idxmax().month - 1]} de {A2}",
                    "Barras: % de ajuste por mes (amarillo = fecha corta, verde = ajuste); abajo, venta bruta y neta",
                    seccion)
    imagen(d, figura_tendencia_cliente(figsize=(15, 7.4)))

    # ------------------------------------------------------------------ 3. materiales
    top1 = top_mat[0]
    share_top1 = mat_cli.loc[(top1, A2), P_GEN] / a2[P_GEN] * 100
    d = diapositiva(f"{str(top1).title()} concentra el {pct(share_top1)} del ajuste, con "
                    f"{pct(mat_cli.loc[(top1, A2), D_FC])} de fecha corta",
                    f"Los {len(top_mat)} materiales con más ajuste general en {PERIODO} {A2}, comparados con {A1}", seccion)
    imagen(d, figura_materiales_cliente(figsize=(15, 7.2)))

    # ------------------------------------------------------------------ 4. mensual
    tm = tabla_mensual_cliente()
    txt = a_texto(tm)
    txt["Facturas"] = tm["Facturas"].map(lambda v: num(v, 0))
    txt.columns = [c.replace(D_AJ, "% Ajuste precio mercado").replace(PRECIO_LISTA, "Lista/kg").replace(PRECIO_NETO, "Neto/kg")
                   for c in tm.columns]
    txt.index.name = "Mes"
    calor = tm[[D_GEN, D_AJ, D_FC]].reindex(columns=tm.columns)
    calor.iloc[-2:] = float("nan")   # los totales no se pintan
    calor.index = txt.index
    d = diapositiva(f"Mes a mes: desde enero de {A2} el ajuste no baja de {pct(m2[D_GEN].min())} y el neto por kilo "
                    f"terminó en {pesos(m2[PRECIO_NETO].iloc[-1])}",
                    "Valores enteros en pesos · mapa de calor en los % de ajuste · al final, los totales del acumulado",
                    seccion)
    tabla(d, txt, y=Inches(1.5), tam=8, ancho_indice=0.11, alto_fila=Inches(0.215),
          negrita_filas=list(txt.index[-2:]), calor=calor)

    guardar(f"Informe_Cliente_{CLIENTE}_{A1}_vs_{A2}.pptx")


construir_informe_cliente()

## C.5 Excel del panorama Región Oriente vs Región Occidente

La tabla de la sección 2 calculada con fórmulas sobre los datos mensuales, para sustentar cada cifra.

In [ ]:
from openpyxl import Workbook
from openpyxl.styles import Alignment, Border, Font, PatternFill, Side
from openpyxl.utils import get_column_letter

EXCEL_PANORAMA = SALIDA / f"Panorama_{A1}_vs_{A2}.xlsx"
ENC = PatternFill("solid", fgColor="F0EFEB")
NEGRITA, GRIS = Font(bold=True), Font(color="52514E", italic=True)
LINEA = Border(bottom=Side(style="thin", color="B5B4AF"))
F_PESOS, F_KG = '"$"#,##0', '#,##0'
F_PCT, F_VAR, F_PTS = '0.0%', '+0.0%;-0.0%', '+0.0" pts";-0.0" pts"'

wb = Workbook()

# --- Datos mensuales (la base de todo)
ws = wb.active
ws.title = "Datos mensuales"
ws.append(["Sociedad", "Año", "Mes", "Venta bruta ($)", "Venta neta ($)", "Venta kg",
           "Ajuste de precio de mercado ($)", "Ajuste de fecha corta ($)", "Ajuste general ($)"])
meses_idx = mensual_soc.index.get_level_values("Mes")
datos = mensual_soc[meses_idx.month.isin(MESES_ACUM) & meses_idx.year.isin(ANIOS)].sort_index()
for (soc, mes), fila in datos.iterrows():
    r = ws.max_row + 1
    ws.append([soc, mes.year, mes.month, fila["Venta Bruta"], fila["Venta Neta"], fila["Venta Kg"],
               fila["Descuento Ajuste Precio Mercado"], fila["Descuento Rotacion / Fecha Corta"], f"=G{r}+H{r}"])
ULT = ws.max_row
for c in ws[1]:
    c.font, c.fill, c.alignment = NEGRITA, ENC, Alignment(horizontal="center", wrap_text=True)
for fila in ws.iter_rows(min_row=2, min_col=4, max_col=9):
    for c in fila:
        c.number_format = F_KG if c.column == 6 else F_PESOS
for col, ancho in zip("ABCDEFGHI", [28, 8, 6, 20, 20, 16, 24, 22, 20]):
    ws.column_dimensions[col].width = ancho
ws.freeze_panes = "A2"
ws.auto_filter.ref = f"A1:I{ULT}"

# --- Panorama con formulas
wp = wb.create_sheet("Panorama", 0)
wp["A1"] = f"Panorama {PERIODO} {A1} vs {A2}: venta bruta, kg y % aplicado al precio de lista"
wp["A1"].font = Font(bold=True, size=14)
wp["A2"] = ("Todas las cifras se calculan con fórmulas sobre la hoja 'Datos mensuales' "
            f"(sumas de enero a {MES_FIN_TXT} de cada año).")
wp["A2"].font = GRIS

D = "'Datos mensuales'!"
COL_DATO = {"bruta": "D", "neta": "E", "kg": "F", "aj": "G", "fc": "H", "gen": "I"}
SUMAS = [("bruta", "Venta bruta ($)", F_PESOS), ("neta", "Venta neta ($)", F_PESOS), ("kg", "Venta kg", F_KG),
         ("aj", "Ajuste de precio de mercado ($)", F_PESOS), ("fc", "Ajuste de fecha corta ($)", F_PESOS),
         ("gen", "Ajuste general ($)", F_PESOS)]
FORMULAS = {
    VENTA_BRUTA: ("={bruta}", F_PESOS, False),
    VENTA_NETA: ("={neta}", F_PESOS, False),
    VENTA_KG: ("={kg}", F_KG, False),
    PRECIO_LISTA: ("={bruta}/{kg}", F_PESOS, False),
    PRECIO_NETO: ("={neta}/{kg}", F_PESOS, False),
    L_GEN: ("=({bruta}-{gen})/{bruta}", F_PCT, True),
    D_GEN: ("={gen}/{bruta}", F_PCT, True),
    P_GEN: ("={gen}", F_PESOS, False),
    L_AJ: ("=({bruta}-{aj})/{bruta}", F_PCT, True),
    D_AJ: ("={aj}/{bruta}", F_PCT, True),
    P_AJ: ("={aj}", F_PESOS, False),
    L_FC: ("=({bruta}-{fc})/{bruta}", F_PCT, True),
    D_FC: ("={fc}/{bruta}", F_PCT, True),
    P_FC: ("={fc}", F_PESOS, False),
}
EXPLICA = {
    VENTA_BRUTA: "Suma de la venta bruta ($)", VENTA_NETA: "Suma de la venta neta ($)",
    VENTA_KG: "Suma de la venta kg", PRECIO_LISTA: "Venta bruta ($) / Venta kg", PRECIO_NETO: "Venta neta ($) / Venta kg",
    L_GEN: "(Venta bruta − Ajuste general) / Venta bruta", D_GEN: "Ajuste general / Venta bruta",
    P_GEN: "Ajuste de precio de mercado + ajuste de fecha corta ($)",
    L_AJ: "(Venta bruta − Ajuste de precio de mercado) / Venta bruta", D_AJ: "Ajuste de precio de mercado / Venta bruta",
    P_AJ: "Suma del ajuste de precio de mercado ($)",
    L_FC: "(Venta bruta − Ajuste de fecha corta) / Venta bruta", D_FC: "Ajuste de fecha corta / Venta bruta",
    P_FC: "Suma del ajuste de fecha corta ($)",
}

FILA_ENC = 4
COL_CALC = 2 + 3 * len(VISTAS)
wp.cell(FILA_ENC + 1, 1, "Concepto")
wp.cell(FILA_ENC + 1, COL_CALC, "Cálculo")
for k, vista in enumerate(VISTAS):
    c0 = 2 + 3 * k
    wp.cell(FILA_ENC, c0, vista)
    wp.merge_cells(start_row=FILA_ENC, start_column=c0, end_row=FILA_ENC, end_column=c0 + 2)
    for j, etiqueta in enumerate([A1, A2, "Variación"]):
        wp.cell(FILA_ENC + 1, c0 + j, etiqueta)
for fila in (FILA_ENC, FILA_ENC + 1):
    for c in wp[fila][:COL_CALC]:
        c.font, c.fill, c.alignment = NEGRITA, ENC, Alignment(horizontal="center")

fila = FILA_ENC + 2
wp.cell(fila, 1, f"Sumas enero–{MES_FIN_TXT} (base de cálculo)").font = NEGRITA
fila_de = {}
for clave, nombre, fmt in SUMAS:
    fila += 1
    fila_de[clave] = fila
    wp.cell(fila, 1, nombre)
    wp.cell(fila, COL_CALC, "Suma de la columna en 'Datos mensuales'"
                            + (" (ajuste + fecha corta)" if clave == "gen" else "")).font = GRIS
    for k, soc in enumerate(VISTAS.values()):
        c0 = 2 + 3 * k
        for j in range(2):
            anio_ref = f"{get_column_letter(c0 + j)}${FILA_ENC + 1}"
            wp.cell(fila, c0 + j, f"=SUMIFS({D}${COL_DATO[clave]}$2:${COL_DATO[clave]}${ULT},"
                                  f"{D}$B$2:$B${ULT},{anio_ref},{D}$A$2:$A${ULT},\"{soc}\")").number_format = fmt
        a_, b_ = get_column_letter(c0), get_column_letter(c0 + 1)
        wp.cell(fila, c0 + 2, f"={b_}{fila}/{a_}{fila}-1").number_format = F_VAR

fila += 2
wp.cell(fila, 1, "Indicadores (tabla de la sección 2)").font = NEGRITA
for ind in BLOQUE:
    fila += 1
    plantilla, fmt, es_pct = FORMULAS[ind]
    wp.cell(fila, 1, ind).font = NEGRITA if ind in (PRECIO_LISTA, PRECIO_NETO, L_GEN, L_AJ, L_FC) else Font()
    wp.cell(fila, COL_CALC, EXPLICA[ind]).font = GRIS
    for k in range(len(VISTAS)):
        c0 = 2 + 3 * k
        for j in range(2):
            col = get_column_letter(c0 + j)
            refs = {clave: f"{col}{fila_de[clave]}" for clave in COL_DATO}
            wp.cell(fila, c0 + j, plantilla.format(**refs)).number_format = fmt
        a_, b_ = get_column_letter(c0), get_column_letter(c0 + 1)
        if es_pct:
            wp.cell(fila, c0 + 2, f"=({b_}{fila}-{a_}{fila})*100").number_format = F_PTS
        else:
            wp.cell(fila, c0 + 2, f"={b_}{fila}/{a_}{fila}-1").number_format = F_VAR
    for c in wp[fila][:COL_CALC - 1]:
        c.border = LINEA

wp.column_dimensions["A"].width = 44
for j in range(2, COL_CALC):
    wp.column_dimensions[get_column_letter(j)].width = 17
wp.column_dimensions[get_column_letter(COL_CALC)].width = 48
wp.freeze_panes = wp.cell(FILA_ENC + 2, 2)

# --- Notas
wn = wb.create_sheet("Notas")
for linea in [
    "Cómo leer",
    "Venta bruta: lo que se habría cobrado a precio de lista (venta neta + ajustes).",
    "Venta neta: lo facturado después de ajustes.",
    "Ajuste general = Ajuste de precio de mercado + Ajuste de fecha corta.",
    "% aplicado al precio de lista + % Ajuste = 100% para cada par (general, ajuste de precio de mercado, fecha corta).",
    "% aplicado al precio de lista ajuste de precio de mercado y fecha corta aíslan un solo ajuste, como si fuera el único.",
    "Variación: % en montos y precios; diferencia en puntos (pts) en los porcentajes.",
    f"Periodo: enero a {MES_FIN_TXT} de {A1} y de {A2}. Datos hasta {FECHA_MAX:%d/%m/%Y}.",
]:
    wn.append([linea])
wn["A1"].font = Font(bold=True, size=12)
wn.column_dimensions["A"].width = 120

try:
    wb.save(EXCEL_PANORAMA)
    print(f"Excel: {EXCEL_PANORAMA.resolve()}")
except PermissionError:
    print(f"No se pudo guardar {EXCEL_PANORAMA.name}: ciérrelo en Excel y vuelva a correr esta celda.")

## C.6 Informe de avance por región

Las mismas láminas de C.2 (portada y secciones 1 a 9) con el mes anterior cerrado vs el mes en curso hasta `CORTE_D`:
la evolución de la sección 2 es día a día y el panorama trae la venta por día.

In [ ]:
def nombre_corte(prefijo: str, vista: str) -> str:
    return (f"{prefijo}_{ARCHIVO[vista]}_{ETQ_ANT.replace(' ', '')}_vs_"
            f"{MES_ACT_TXT[:3].capitalize()}{INICIO_ACT.year}_al_{FECHA_MAX_D.day}.pptx")


def portada_corte(titulo: str, linea_2: str, soc):
    portada(titulo, f"{MES_ANT_TXT.capitalize()} {INICIO_ANT.year} (cerrado) vs {MES_ACT_TXT} al {FECHA_MAX_D.day} "
                    f"(en curso) · {nombre_sociedad(soc)}", f"{linea_2} · datos a {FECHA_MAX_D:%d/%m/%Y}",
            COLOR_SOC[soc])


def construir_informe_corte(vista: str):
    """El informe general (C.2) con el mes anterior cerrado vs el mes en curso."""
    soc = VISTAS[vista]
    with modo_corte():
        nueva_presentacion()

        # ------------------------------------------------------------------ portada
        portada_corte("Diagnóstico de Ajustes y % Cumplimiento Aplicado a Lista de Precios",
                      "Todos los canales · avance del mes", soc)

        # ------------------------------------------------------------------ 1. contexto del alcance
        d = prs.slides.add_slide(prs.slide_layouts[6])
        caja_texto(d, MARGEN, Inches(0.46), UTIL_W, Inches(0.37), sec(1).upper(), tam=16, color=ACENTO, negrita=True)
        caja_texto(d, MARGEN, Inches(1.0), UTIL_W, Inches(0.57), ALCANCE, tam=14)
        pilar = pd.DataFrame([
            ["neta / bruta", "ajuste + fecha corta"],
            ["(bruta − ajuste) / bruta", "solo el ajuste de precio de mercado"],
            ["(bruta − fecha corta) / bruta", "solo la fecha corta (rotación)"],
        ], index=["% Aplicado al precio de lista general  |  % Ajuste general",
                  "% Aplicado al precio de lista por ajuste de precio de mercado  |  % Ajuste de precio de mercado",
                  "% Aplicado al precio de lista por fecha corta  |  % Ajuste de fecha corta"],
           columns=["% de lista precio", "% de ajuste"])
        pilar.index.name = "El pilar: % Aplicado al precio de lista + % de ajuste = 100%"
        tabla(d, pilar, y=Inches(2.05), tam=12, ancho_indice=0.46, alto_fila=Inches(0.42))
        caja_texto(d, MARGEN, Inches(4.31), UTIL_W, Inches(1.9), [
            [("Venta bruta: ", True), ("lo que se habría cobrado a precio de lista (venta neta + ajustes). Es la base "
                                       "de todas las participaciones.", False)],
            [("Venta neta: ", True), ("Valor facturado;", False)],
            [("Venta kg: ", True), ("kilos vendidos.", False)],
            [("Rangos: ", True), ("22 rangos del % Aplicado al precio de lista Cada venta (día × cliente × grupo de "
                                  "material × distribuidora) cae en uno.", False)],
            [("Tipos de Ajuste: ", True), ("Ajuste de fecha corta (rotación) y ajuste de precio de mercado", False)],
            [("Periodos: ", True), (f"{MES_ANT_TXT} {INICIO_ANT.year} completo (mes cerrado) vs {MES_ACT_TXT} del 1 "
                                    f"al {FECHA_MAX_D.day} (mes en curso). Los kg y pesos del mes en curso son "
                                    "parciales: compare los % y los precios por kg.", False)],
        ], tam=13, espacio=7)

        # ------------------------------------------------------------------ 2. panorama y evolucion dia a dia
        pan = a_texto(PANORAMA[vista], filas_indicador=True)
        pan.index.name = "Indicador"
        d = lamina(f"La lista por kilo {sube_baja(ch_(vista, PRECIO_LISTA))} {num(abs(ch_(vista, PRECIO_LISTA)))}%, "
                   f"y el neto por kilo {sube_baja(ch_(vista, PRECIO_NETO))} {num(abs(ch_(vista, PRECIO_NETO)))}%",
                   f"% aplicado al precio de lista general {de_a(vista, L_GEN)} · ajuste de precio de mercado "
                   f"{de_a(vista, L_AJ)} · fecha corta {de_a(vista, L_FC)}", sec(2))
        tabla(d, pan, x=Inches(1.5), w=Inches(10.3), tam=13, ancho_indice=0.46, alto_fila=Inches(0.31),
              negrita_filas=[PRECIO_LISTA, PRECIO_NETO, L_GEN, L_AJ, L_FC, KG_DIA, BRUTA_DIA])
        for paneles, titulo in ((("kg", "pesos"), None),
                                (("precio", "aplicado"),
                                 f"El % aplicado al precio de lista general pasó de {pct(v_(vista, L_GEN, A1))} "
                                 f"({A1}) a {pct(v_(vista, L_GEN, A2))} ({A2})")):
            d = lamina(titulo, seccion=sec(2))
            fig = figura_evolucion(soc, paneles, figsize=(15, 7.2))
            fig.suptitle(f"{vista}: evolución día a día", x=0.01, ha="left", fontsize=12)
            imagen(d, fig, y=Inches(1.15 if titulo else 1.0), h=Inches(5.75 if titulo else 5.8))

        # ------------------------------------------------------------------ 3 a 6. rangos
        lamina_rangos("Total", vista, soc, "Total", sec(3))
        for dim, n in SECCION_DIM.items():
            lamina_resumen_dim(dim, vista, soc, sec(n))
            tope = TOP_PPT if dim == "Grupo de material 6" else None
            for m in miembros(dim, soc)[:tope]:
                lamina_rangos(dim, vista, soc, m, sec(n))

        # ------------------------------------------------------------------ 7. canal de venta x canal de distribucion
        for dim in CRUCES:
            for m in miembros(dim, soc):
                t = desglose_canal(dim, soc, m)
                canales = [x for x in t.columns.unique(1) if x != "Total"]
                kg2 = t.loc[VENTA_KG, A2]
                if len(canales) > 1:
                    mayor = max(canales, key=lambda x: kg2.get(x, 0) if pd.notna(kg2.get(x)) else 0)
                    otro = next(x for x in canales if x != mayor)
                    titulo = (f"{m}: {mayor} es el {pct(kg2[mayor] / kg2['Total'] * 100)} de la venta kg; "
                              f"% aplicado al precio de lista general {pct(t.loc[L_GEN, (A2, mayor)])} en {mayor} "
                              f"y {pct(t.loc[L_GEN, (A2, otro)])} en {otro} ({A2})")
                else:
                    titulo = f"{m}: se vende solo por {canales[0]}"
                txt = a_texto(t, filas_indicador=True)
                txt.index.name = "Indicador"
                d = lamina(titulo, f"{vista} · {dim} · Mayoreo + Puntos de Venta = Total (kg, venta y ajustes); "
                                   f"los % y precios se recalculan · {A1} vs {A2}", sec(7))
                tabla(d, txt, tam=10, ancho_indice=0.27, alto_fila=Inches(0.33), negrita_filas=[L_GEN, L_AJ, L_FC])

        # ------------------------------------------------------------------ 8. distribuidoras
        mitad = mitad_distribuidoras(soc)
        d = lamina(f"{len(mitad)} distribuidoras concentran la mitad del ajuste general — "
                   + ", ".join(n_.title() for n_ in mitad[:6]),
                   f"{vista} · ordenadas por ajuste general de {A2}", sec(8))
        tabla(d, tabla_conc(distribuidoras(soc).head(FILAS_PPT), "Distribuidora"), tam=9,
              negrita_filas=[str(i)[:40] for i in mitad], alto_fila=Inches(0.33), anchos=ANCHOS_CONC)

        # ------------------------------------------------------------------ 9. clientes del mes en curso
        for canal in CANALES:
            tc = top_clientes(soc, canal)
            if tc.empty:
                continue
            cc = concentracion(soc, canal)
            n_ = cc.loc["Clientes con el 50% del ajuste general", A2]
            txt = a_texto(tc)
            txt.columns = [corto(col) for col in tc.columns]
            txt.index = [str(i)[:38] for i in txt.index]
            txt.index.name = "Cliente"
            d = lamina(f"Clientes {canal}: {num(n_, 0)} concentran la mitad del ajuste general de {A2}",
                       f"{vista} · top {TOP_CLIENTES} por ajuste general en {A2} · "
                       f"{num(cc.loc['Clientes con ajuste', A2], 0)} clientes con ajuste "
                       f"({num(cc.loc['Clientes con ajuste', A1], 0)} en {A1})",
                       f"9. Concentración del ajuste – Clientes {ROTULO_CANAL.get(canal, canal)} – {A2}")
            tabla(d, txt, y=Inches(1.5), tam=7.5, ancho_indice=0.22, alto_fila=Inches(0.166),
                  calor=calor_ajustes(tc, txt.index))

        guardar(nombre_corte("Informe_Avance", vista))

### Región Oriente

In [ ]:
construir_informe_corte("Región Oriente")

### Región Occidente

In [ ]:
construir_informe_corte("Región Occidente")

## C.7 Foco Res por canal de venta

Un PowerPoint por región y canal de venta (`CANALES_VENTA_TOP`: 8 en total) para los materiales de `FOCO`, mes
anterior cerrado vs mes en curso al corte. Por material:

| Láminas | Contenido |
|---|---|
| General | Tabla del canal (Mayoreo + Puntos de Venta = Total del canal) y la gráfica de rangos del canal |
| Mayoreo / Puntos de Venta | La gráfica de rangos de cada canal de distribución con venta |
| Top distribuidoras | Las `TOP_DISTRIBUIDORAS` con más ajuste general, con su % acumulado |
| Top clientes | Los `TOP_CLIENTES` con más ajuste general, con su canal, distribuidora y el % de ajuste del mes anterior |

In [ ]:
def lamina_foco_general(vista, soc, m, cv, seccion, nota):
    """Tabla del material en el canal de venta: Mayoreo + Puntos de Venta = Total del canal."""
    t = foco_canal_cd(soc, m)
    tot = t.loc["Total"]
    abiertas = [i for i in t.index if i != "Total"]
    kg = t[(VENTA_KG, A2)]
    mayor = kg[abiertas].idxmax() if kg[abiertas].notna().any() else None
    d = lamina(f"{m} · {cv}: % aplicado al precio de lista general {pct(tot[(L_GEN, A1)])} → {pct(tot[(L_GEN, A2)])}",
               f"{vista} · " + (f"{mayor} es el {pct(kg[mayor] / kg['Total'] * 100)} de la venta kg de {A2} · "
                                if mayor else "") + f"Mayoreo + Puntos de Venta = Total del canal · {nota}", seccion)
    txt = a_texto(t)
    txt.columns = pd.MultiIndex.from_tuples([(corto(i), a_) for i, a_ in t.columns])
    tabla(d, txt, y=Inches(1.7), tam=10, ancho_indice=0.14, alto_fila=Inches(0.4), calor=calor_ajustes(t, txt.index),
          negrita_filas=["Total"])


def lamina_foco_rangos(vista, soc, m, seccion, canal_dist=None, canal_venta=None):
    """Grafica de rangos (como la seccion 3 del informe general) del material en el canal de venta, y si se da, en un
    canal de distribucion."""
    c = foco_de(soc, m)
    nombre = m
    if canal_venta:
        c = c[c.index.get_level_values("Canal de Venta") == canal_venta]
        nombre += f" · {canal_venta}"
    if canal_dist:
        c = c[c.index.get_level_values("Canal de Distribucion") == canal_dist]
        nombre += f" · {ROTULO_CANAL.get(canal_dist, canal_dist)}"
    ind = indicadores(c.groupby(level="Año").sum()).reindex(ANIOS)
    bajada = (f"{vista} · % aplicado al precio de lista general {pct(ind.loc[A1, L_GEN])} → "
              f"{pct(ind.loc[A2, L_GEN])} · ajuste de precio de mercado {pct(ind.loc[A1, L_AJ])} → "
              f"{pct(ind.loc[A2, L_AJ])} · fecha corta {pct(ind.loc[A1, L_FC])} → {pct(ind.loc[A2, L_FC])} · venta kg "
              f"{num(ind.loc[A1, VENTA_KG], 0)} → {num(ind.loc[A2, VENTA_KG], 0)}")
    d = lamina(f"{nombre}: {frase_rango(c)}", bajada, seccion)
    imagen(d, figura_rangos(c, f"% de la venta bruta por rango de % aplicado al precio de lista general y motivo — "
                               f"{nombre} (al lado de cada barra: % y ajuste general)", figsize=(14, 7.4)))


def lamina_foco_distribuidoras(vista, soc, m, seccion, filas=TOP_DISTRIBUIDORAS):
    t = foco_distribuidoras(soc, m)
    a2 = str(A2)   # el pareto rotula los periodos como texto
    if t.empty or not t[(P_GEN, a2)].gt(0).any():
        lamina(f"{m}: sin ajuste general en {A2}", vista, seccion)
        return
    mitad = list(t.index[: int((t[("% acumulado ajuste general", a2)] < 50).sum()) + 1])
    top = t.head(filas)
    d = lamina(f"{m}: {len(mitad)} distribuidoras concentran la mitad del ajuste general de {A2} — "
               + ", ".join(n_.title() for n_ in mitad[:6]),
               f"{vista} · top {len(top)} de {len(t)} distribuidoras con venta, ordenadas por ajuste general de {A2} · "
               f"ajuste general: {pesos(t[(P_GEN, a2)].sum())}", seccion)
    tabla(d, tabla_conc(top, "Distribuidora"), tam=9 if len(top) <= 14 else 7.5,
          negrita_filas=[str(i)[:40] for i in mitad], alto_fila=Inches(min(0.33, 5.4 / (len(top) + 2))),
          anchos=ANCHOS_CONC)


def lamina_foco_clientes(vista, soc, m, seccion):
    tc, part, total, n_cli = foco_clientes(soc, m)
    if tc is None:
        lamina(f"{m}: sin clientes con ajuste en {A2}", vista, seccion)
        return
    txt = a_texto(tc)
    txt.columns = [corto(c_) for c_ in tc.columns]
    txt.index = [str(i)[:34] for i in txt.index]
    txt.index.name = "Cliente"
    sujeto = ("el único cliente con ajuste concentra" if len(tc) == 1
              else f"los {len(tc)} clientes con más ajuste general suman")
    d = lamina(f"{m}: {sujeto} el {pct(part)} del ajuste general de {A2}",
               f"{vista} · ajuste general: {pesos(total)} · {num(n_cli, 0)} clientes con ajuste · "
               f"ordenados por ajuste general en pesos · al lado, el % de ajuste general de {A1}", seccion)
    tabla(d, txt, y=Inches(1.5), tam=7, ancho_indice=0.19, alto_fila=Inches(0.17), calor=calor_ajustes(tc, txt.index))


def laminas_foco_canal(vista, soc, cv, nota):
    """Por material, en el canal de venta cv (foco_d ya filtrado con solo_canal): tabla general del canal, grafica de
    rangos del canal, una grafica por canal de distribucion, top de distribuidoras y top de clientes."""
    texto = f"{vista} · {cv}"
    for n, m in enumerate(FOCO, start=1):
        sec_ = f"{n}. {m} · {cv}"
        if foco_de(soc, m)["Venta Bruta"].sum() <= 0:
            lamina(f"{m}: sin venta en {cv}", texto, sec_)
            continue
        lamina_foco_general(texto, soc, m, cv, f"{sec_} · General", nota)
        lamina_foco_rangos(texto, soc, m, f"{sec_} · General", canal_venta=cv)
        for cd in canales_dist_foco(soc, m):
            lamina_foco_rangos(texto, soc, m, f"{sec_} · {ROTULO_CANAL.get(cd, cd)}", canal_dist=cd, canal_venta=cv)
        lamina_foco_distribuidoras(texto, soc, m, f"{sec_} · Top distribuidoras")
        lamina_foco_clientes(texto, soc, m, f"{sec_} · Top clientes")


def informe_foco_res(vista: str, cv: str):
    """Version mes anterior cerrado vs mes en curso, un canal de venta."""
    soc = VISTAS[vista]
    with modo_corte(), solo_canal(cv):
        nueva_presentacion()
        portada_corte(f"Foco Res · {cv}: {' y '.join(FOCO)}",
                      "General del canal, Mayoreo, Puntos de Venta y top de distribuidoras y clientes", soc)
        dias = dias_con_venta(soc)
        laminas_foco_canal(vista, soc, cv, f"kg y pesos de {A2} parciales ({dias[A2]} días con venta frente a "
                                           f"{dias[A1]})")
        guardar(nombre_corte("Foco_Res", vista).replace(ARCHIVO[vista], f"{ARCHIVO[vista]}_{cv.replace(' ', '_')}"))


for vista in VISTAS:
    for cv in CANALES_VENTA_TOP:
        informe_foco_res(vista, cv)